# Comparing AltiKa tracks to simple wave models



## Remove tracks that are physically impossible

In [ ]:
from intake import cat
from xarray import DataTree, map_over_datasets
from distributed import Client
import glob
import xarray as xr
import cf_xarray
import numpy as np
from datetime import timedelta
import cf_xarray as cfxr
import xesmf
import re
import os
import time
import intake
from tqdm.notebook import tqdm


# Plotting
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean.cm as cmo
import matplotlib.lines as mlines
import cartopy.feature as cft
import seaborn as sns

# Import my functions
functions_path = os.path.abspath("/home/566/nd0349/access-om3-analysis/functions")
if functions_path not in sys.path:
    sys.path.append(functions_path)
from get_files import *
from plot_settings import *
from fstd import *
from parameters import *
from altika import *
from stats import *
from wave_propagation import *
from attenuation_models import *
π = np.pi
test()

# --- Canonical column rename map and helper to keep names consistent ---
rename_map = {
    "first_meas_time": "timestamp",
    "swhAtMyEdge": "hs_my_edge",
    "lonAtMyEdge": "lon_my_edge",
    "latAtMyEdge": "lat_my_edge",
    "lonAtAltiKaEdge": "lon_altika_edge",
    "latAtAltiKaEdge": "lat_altika_edge",
    "lonAtInnerMIZ": "lon_inner_miz",
    "latAtInnerMIZ": "lat_inner_miz",
    "mizWidthAlongTrackFromMyEdge": "miz_width_myedge_km",
    "mizWidthAlongTrackFromAltikaEdge": "miz_width_altikaedge_km",
    "mizwidth_lat": "miz_width_lat_km",
    "swhAtInnerEdge": "hs_inner_edge",
    "swhAtInnerEdge_aice1": "hs_inner_edge_aice1",
    "swhFraction": "hs_fraction",
    "distance_until_swh_fraction_km": "distance_until_hs_fraction_km",
    "est_miz_width_altika_km": "est_miz_width_altika_km",
    "est_miz_width_altika_eff_km": "est_miz_width_altika_eff_km",
    "est_miz_width_ww3_km": "est_miz_width_ww3_km",
    "est_miz_width_ww3_eff_km": "est_miz_width_ww3_eff_km",
    "est_miz_width_altika_aice1_km": "est_miz_width_altika_aice1_km",
    "est_miz_width_altika_aice1_eff_km": "est_miz_width_altika_aice1_eff_km",
    "est_miz_width_ww3_tp_km": "est_miz_width_ww3_tp_km",
    "est_miz_width_ww3_tp_eff_km": "est_miz_width_ww3_tp_eff_km",
    "est_miz_width_ww3_south_km": "est_miz_width_ww3_south_km",
    "est_miz_width_ww3_south_eff_km": "est_miz_width_ww3_south_eff_km",
    "ICE_track_mean_myedge": "ice_track_mean_myedge",
    "TP_edge_myedge": "tp_edge_myedge",
    "HS_edge_myedge": "hs_edge_myedge",
    "TP_edge_north_myedge": "tp_edge_north_myedge",
    "HS_edge_north_myedge": "hs_edge_north_myedge",
    "ww3_miz_width_hs05_km_myedge": "ww3_miz_width_hs05_km_myedge",
    "ww3_miz_width_hs05_km_myedge_eff": "ww3_miz_width_hs05_km_myedge_eff",
}

def normalize_df(df):
    """Return a dataframe with canonical column names applied (inplace-safe)."""
    if df is None:
        return df
    # only rename columns that exist to avoid KeyErrors
    existing = {k: v for k, v in rename_map.items() if k in df.columns}
    return df.rename(columns=existing)

- first_meas_time: AltiKa measurement timestamp. Units: datetime.
- hs_my_edge: AltiKa significant wave height at the user-defined / “my” ice edge. Units: m.
- lonAtMyEdge: Longitude of the user-defined / “my” ice edge. Units: degrees east.
- latAtMyEdge: Latitude of the user-defined / “my” ice edge. Units: degrees north.
- lonAtAltiKaEdge: Longitude of the AltiKa-derived ice edge. Units: degrees east.
- latAtAltiKaEdge: Latitude of the AltiKa-derived ice edge. Units: degrees north.
- lonAtInnerMIZ: Longitude of the inner MIZ boundary. Units: degrees east.
- latAtInnerMIZ: Latitude of the inner MIZ boundary. Units: degrees north.
- mizWidthAlongTrackFromMyEdge: Satellite-track MIZ width measured from `MyEdge` to `InnerMIZ`. Units: km.
- mizWidthAlongTrackFromAltikaEdge: Satellite-track MIZ width measured from `AltiKaEdge` to `InnerMIZ`. Units: km.
- date: Calendar date extracted from `first_meas_time`. Units: date.
- day: Day of month extracted from `first_meas_time`. Units: day number.
- year: Year extracted from `first_meas_time`. Units: year number.
- month: Month extracted from `first_meas_time`. Units: month number.
- altika_myedge_dist_km: Great-circle distance between `AltiKaEdge` and `MyEdge`. Units: km.
- ww3_time: Nearest WW3 model timestamp matched to `first_meas_time`. Units: datetime.
- ww3_time_idx: Integer time index of `ww3_time` in the WW3 dataset. Units: index.

- ww3_edge_y_altikaedge: WW3 grid y-index nearest to the AltiKa-derived edge. Units: grid index.
- ww3_edge_x_altikaedge: WW3 grid x-index nearest to the AltiKa-derived edge. Units: grid index.
- ww3_inner_y_altikaedge: WW3 grid y-index nearest to the inner MIZ boundary along the AltiKa-edge track. Units: grid index.
- ww3_inner_x_altikaedge: WW3 grid x-index nearest to the inner MIZ boundary along the AltiKa-edge track. Units: grid index.
- ww3_track_grid_cell_count_altikaedge: Number of unique WW3 grid cells sampled along the AltiKa-edge-to-inner-MIZ track. Units: count.
- ICE_track_mean_altikaedge: Mean WW3 sea ice concentration along the AltiKa-edge-to-inner-MIZ track. Units: nondimensional fraction.
- ICE_track_min_altikaedge: Minimum WW3 sea ice concentration along the AltiKa-edge-to-inner-MIZ track. Units: nondimensional fraction.
- ICE_track_max_altikaedge: Maximum WW3 sea ice concentration along the AltiKa-edge-to-inner-MIZ track. Units: nondimensional fraction.
- ICE_edge_altikaedge: WW3 sea ice concentration at the AltiKa-derived edge. Units: nondimensional fraction.
- ICE_inner_altikaedge: WW3 sea ice concentration at the inner MIZ boundary for the AltiKa-edge track. Units: nondimensional fraction.
- HS_edge_altikaedge: WW3 significant wave height at the AltiKa-derived edge. Units: m.
- HS_inner_altikaedge: WW3 significant wave height at the inner MIZ boundary for the AltiKa-edge track. Units: m.
- THM_edge_altikaedge: WW3 mean wave direction at the AltiKa-derived edge. Units: radians.
- THM_inner_altikaedge: WW3 mean wave direction at the inner MIZ boundary for the AltiKa-edge track. Units: radians.
- UAX_edge_altikaedge: WW3 mean wind x-component at the AltiKa-derived edge. Units: m s^-1.
- UAX_inner_altikaedge: WW3 mean wind x-component at the inner MIZ boundary for the AltiKa-edge track. Units: m s^-1.
- UAY_edge_altikaedge: WW3 mean wind y-component at the AltiKa-derived edge. Units: m s^-1.
- UAY_inner_altikaedge: WW3 mean wind y-component at the inner MIZ boundary for the AltiKa-edge track. Units: m s^-1.
- TM02_edge_altikaedge: WW3 mean wave period `T02`, saved as `TM02`, at the AltiKa-derived edge. Units: s.
- TM02_inner_altikaedge: WW3 mean wave period `T02`, saved as `TM02`, at the inner MIZ boundary for the AltiKa-edge track. Units: s.
- TP_edge_altikaedge: WW3 peak wave period at the AltiKa-derived edge, computed as `1 / FP0`. Units: s.
- TP_inner_altikaedge: WW3 peak wave period at the inner MIZ boundary for the AltiKa-edge track, computed as `1 / FP0`. Units: s.

- ww3_edge_y_myedge: WW3 grid y-index nearest to the user-defined / “my” ice edge. Units: grid index.
- ww3_edge_x_myedge: WW3 grid x-index nearest to the user-defined / “my” ice edge. Units: grid index.
- ww3_inner_y_myedge: WW3 grid y-index nearest to the inner MIZ boundary along the MyEdge-to-inner-MIZ track. Units: grid index.
- ww3_inner_x_myedge: WW3 grid x-index nearest to the inner MIZ boundary along the MyEdge-to-inner-MIZ track. Units: grid index.
- ww3_track_grid_cell_count_myedge: Number of unique WW3 grid cells sampled along the MyEdge-to-inner-MIZ track. Units: count.
- ICE_track_mean_myedge: Mean WW3 sea ice concentration along the MyEdge-to-inner-MIZ track. Units: nondimensional fraction.
- ICE_track_min_myedge: Minimum WW3 sea ice concentration along the MyEdge-to-inner-MIZ track. Units: nondimensional fraction.
- ICE_track_max_myedge: Maximum WW3 sea ice concentration along the MyEdge-to-inner-MIZ track. Units: nondimensional fraction.
- ICE_edge_myedge: WW3 sea ice concentration at the user-defined / “my” ice edge. Units: nondimensional fraction.
- ICE_inner_myedge: WW3 sea ice concentration at the inner MIZ boundary for the MyEdge track. Units: nondimensional fraction.
- HS_edge_north_myedge: WW3 significant wave height at the user-defined / “my” ice edge. Units: m.
- HS_inner_myedge: WW3 significant wave height at the inner MIZ boundary for the MyEdge track. Units: m.
- THM_edge_myedge: WW3 mean wave direction at the user-defined / “my” ice edge. Units: radians.
- THM_inner_myedge: WW3 mean wave direction at the inner MIZ boundary for the MyEdge track. Units: radians.
- UAX_edge_myedge: WW3 mean wind x-component at the user-defined / “my” ice edge. Units: m s^-1.
- UAX_inner_myedge: WW3 mean wind x-component at the inner MIZ boundary for the MyEdge track. Units: m s^-1.
- UAY_edge_myedge: WW3 mean wind y-component at the user-defined / “my” ice edge. Units: m s^-1.
- UAY_inner_myedge: WW3 mean wind y-component at the inner MIZ boundary for the MyEdge track. Units: m s^-1.
- TM02_edge_myedge: WW3 mean wave period `T02`, saved as `TM02`, at the user-defined / “my” ice edge. Units: s.
- TM02_inner_myedge: WW3 mean wave period `T02`, saved as `TM02`, at the inner MIZ boundary for the MyEdge track. Units: s.
- TP_edge_myedge: WW3 peak wave period at the user-defined / “my” ice edge, computed as `1 / FP0`. Units: s.
- TP_inner_myedge: WW3 peak wave period at the inner MIZ boundary for the MyEdge track, computed as `1 / FP0`. Units: s.

- ww3_miz_width_hs044_km_altikaedge: WW3-estimated MIZ width along the AltiKa-edge track, defined as the along-track distance from the AltiKa-derived edge to the first point where WW3 `HS < 0.44 m`. Units: km.
- ww3_miz_width_hs05_found_altikaedge: Flag for whether `HS < 0.44 m` was found along the AltiKa-edge track; `1` means found, `0` means not found. Units: binary flag.
- ww3_miz_width_hs05_km_myedge: WW3-estimated MIZ width along the MyEdge track, defined as the along-track distance from the user-defined / “my” ice edge to the first point where WW3 `HS < 0.44 m`. Units: km.
- ww3_miz_width_hs05_found_myedge: Flag for whether `HS < 0.44 m` was found along the MyEdge track; `1` means found, `0` means not found. Units: binary flag.

In [ ]:
client = Client(threads_per_worker=1)
client

In [ ]:
print(client.dashboard_link)

## Read in AltiKa data

In [ ]:
from tqdm.notebook import tqdm

years = range(2013, 2024)
dfs = []

# swh_min = 10**-12
# swh_max = 100
# miz_max = 10000
# miz_min = -10**-12

for year in tqdm(years):
    # print(year)
    df = pd.DataFrame(ReadInAltika(version='0.15', year=year))
    df = normalize_df(df)
    df = df[['timestamp', 'hs_my_edge', 'lon_my_edge', 'lat_my_edge', 'lon_altika_edge','lat_altika_edge', 'lon_inner_miz', 'lat_inner_miz', 'miz_width_myedge_km', 'miz_width_altikaedge_km']]
    df['date'] = pd.to_datetime(df["timestamp"]).dt.date # , format='%Y-%m-%d %H:%M:%S.%f').dt.date
    df['day'] = pd.to_datetime(df['date']).dt.day
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    df['miz_width_lat_km'] = abs(df['lat_altika_edge'] - df['lat_inner_miz'])*111.32 # Alex's conversion from latitudes to km

    dfs.append(df)
# Combine into one dataframe
df_all = pd.concat(dfs, ignore_index=True)
# Normalize column names to canonical names
df_all = normalize_df(df_all)
# df_tmp = df_all.copy()
# condition_met_df = (df_tmp['hs_my_edge'] < swh_max) & (df_tmp['hs_my_edge'] > swh_min) & (df_tmp['miz_width_altikaedge_km'] < miz_max) & (df_tmp['miz_width_altikaedge_km'] > miz_min)
# df_all = df_tmp.where(condition_met_df)
df_all.head()

In [ ]:
files = sorted(glob.glob("/g/data/ps29/nd0349/Fraser-2024/data/cleaned/*_effective_hourly_50_v0_15.csv"))
# print(files)
df_eff = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)
# normalize column names
df_eff = normalize_df(df_eff)
df_eff

In [ ]:
files = sorted(glob.glob("/g/data/ps29/nd0349/Fraser-2024/data/cleaned/*_altika_tracks_on_ww3_both_edges_n30_v0_15.csv"))
# print(files)
df_eff = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)
# normalize column names
df_eff = normalize_df(df_eff)
df_eff

## Attenuation models

In [ ]:
freqs, periods = get_ww3_freqs(nk=25)
α_mbk = mbk_2014_attenuation(periods)
thickness = 0.5
floeDiameter = 1000
α_meylan, _ = meylan_2021_attenuation(periods, thickness, floeDiameter)

In [ ]:
plt.plot(periods, α_mbk, label="IC4M2 (Meylan et al., 2014)")
plt.plot(periods, α_meylan, label="IC4M8 (Meylan et al., 2021)")
plt.yscale('log')
plt.xlabel("Period [s]")
plt.ylabel("Attenuation coefficient [1/m]")
plt.legend()

In [ ]:
hs = 2
tm = 16
omega = 2 * np.pi * freqs
S_om, t = sdf_bretschneider(hs, tm, omega)

In [ ]:
m0_truncated = np.trapz(S_om, omega)

hs_recovered = 4 * np.sqrt(m0_truncated)

m0_expected = (hs / 4) ** 2

energy_fraction = m0_truncated / m0_expected
energy_truncated_fraction = 1 - energy_fraction

print(f"m0 expected: {m0_expected:.4f}")
print(f"m0 recovered: {m0_truncated:.4f}")
print(f"Hs recovered: {hs_recovered:.3f} m")
print(f"Energy retained: {100 * energy_fraction:.2f}%")
print(f"Energy truncated: {100 * energy_truncated_fraction:.2f}%")

In [ ]:
plt.plot(t, S_om)

## Propagate waves

In [ ]:
x, hs_x = advect_spectrum_hs(
    freqs=freqs,
    S_om=S_om,
    x_max=500e3,
    dx=1e3,
    alpha_units="m-1",
)

In [ ]:
plt.plot(x / 1000, hs_x)
plt.xlabel("Distance [km]")
plt.ylabel("$H_s$ [m]")
plt.grid(True)
plt.show()

## Distance until critical $H_s$

In [ ]:
hs0_values = np.arange(0.5, 8.5, 0.5)
tp_values = np.arange(5, 26, 1)

distance_to_threshold = np.full(
    (len(hs0_values), len(tp_values)),
    np.nan,
    dtype=float
)

for i, hs0 in enumerate(hs0_values):
    for j, tp in enumerate(tp_values):
        distance, _, _ = distance_until_hs_below(
            hs0=hs0,
            tp=tp,
            freqs=freqs,
            aice=1.0,
            hs_threshold=0.01,
            x_max=10000e3,
            dx=1e3,
            alpha_units="m-1",
        )

        distance_to_threshold[i, j] = distance

In [ ]:
distance_to_threshold_km = distance_to_threshold / 1000

In [ ]:
Tp_grid, Hs_grid = np.meshgrid(tp_values, hs0_values)

plt.figure(figsize=(7, 4))
cs = plt.contourf(Tp_grid, Hs_grid, distance_to_threshold_km, levels=20)
plt.colorbar(cs, label="Distance until $H_s < 0.01$ m [km]")
plt.xlabel("$T_p$ [s]")
plt.ylabel("Initial $H_s$ [m]")
plt.show()

In [ ]:
df_sample = df_eff.sample(n=10000, #random_state=42
                         ).copy()

df_sample["distance_until_hs_below_km"] = df_sample["hs_my_edge"].apply(
    threshold_distance_from_hs
)

In [ ]:
df_sample.columns

In [ ]:
dist_col = "distance_until_hs_below_km"
miz_col = "miz_width_myedge_km"

x1 = df_sample[dist_col].replace([np.inf, -np.inf], np.nan).dropna()
x2 = df_sample[miz_col].replace([np.inf, -np.inf], np.nan).dropna()

if len(x1) == 0:
    raise ValueError(f"No valid values in {dist_col}")

if len(x2) == 0:
    raise ValueError(f"No valid values in {miz_col}")

xmin = min(x1.min(), x2.min())
xmax = max(x1.max(), x2.max())

if xmin == xmax:
    bins = 20
else:
    bins = np.linspace(xmin, xmax, 50)

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(
    x1,
    bins=bins,
    stat="density",
    kde=len(x1) > 1,
    alpha=0.35,
    label="Attenuation distance",
    ax=ax
)

sns.histplot(
    x2,
    bins=bins,
    stat="density",
    kde=len(x2) > 1,
    alpha=0.35,
    label="Observed MIZ width",
    ax=ax
)

ax.set_xlabel("Distance / width [km]")
ax.set_ylabel("Density")
ax.legend()
ax.grid(True)

plt.show()

In [ ]:
dist_col = "distance_until_hs_below_km"
miz_col = "miz_width_myedge_km"

df_reduced = df_sample[
    df_sample[dist_col].notna()
    & df_sample[miz_col].notna()
    & (df_sample[dist_col] >= df_sample[miz_col])
].copy()

In [ ]:
df_sample['hs_my_edge'].hist()

In [ ]:
df_reduced['hs_my_edge'].hist()

In [ ]:
miz_col = "miz_width_myedge_km"
hs_col = "hs_my_edge"

x_original = df_sample[miz_col].dropna()
x_reduced = df_reduced[miz_col].dropna()

bins = np.linspace(
    min(x_original.min(), x_reduced.min()),
    max(x_original.max(), x_reduced.max()),
    50
)

fig, axes = plt.subplots(
    1, 4,
    figsize=(15, 4),
    constrained_layout=True,
    # gridspec_kw={"width_ratios": [1.4, 0.8, 1.4]}
)

# ---------------- Histogram + KDE ----------------
ax = axes[0]

sns.histplot(
    x_original,
    bins=bins,
    stat="density",
    kde=True,
    alpha=0.30,
    label=f"Original, n={len(x_original)}",
    ax=ax
)

sns.histplot(
    x_reduced,
    bins=bins,
    stat="density",
    kde=True,
    alpha=0.30,
    label=f"Without unphysical tracks, n={len(x_reduced)}",
    ax=ax
)

ax.set_xlabel("MIZ width [km]")
ax.set_ylabel("Density")
ax.legend()
ax.grid(True)


# ---------------- Boxplot ----------------
ax = axes[1]

box_df = pd.concat([
    pd.DataFrame({
        "Dataset": "Original",
        "MIZ width [km]": x_original.values
    }),
    pd.DataFrame({
        "Dataset": "Without unphysical tracks",
        "MIZ width [km]": x_reduced.values
    }),
])

sns.boxplot(
    data=box_df,
    x="Dataset",
    y="MIZ width [km]",
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("MIZ width [km]")
ax.tick_params(axis="x", rotation=25)
ax.grid(True, axis="y")


# ---------------- Scatter + best fit ----------------
ax = axes[2]

df_original_plot = df_sample[[hs_col, miz_col]].dropna()
df_reduced_plot = df_reduced[[hs_col, miz_col]].dropna()

r_original = df_original_plot[hs_col].corr(df_original_plot[miz_col])
r_reduced = df_reduced_plot[hs_col].corr(df_reduced_plot[miz_col])

sns.regplot(
    data=df_original_plot,
    x=hs_col,
    y=miz_col,
    ax=ax,
    scatter_kws={"s": 18, "alpha": 0.25},
    line_kws={"linewidth": 2},
    label=f"Original, r={r_original:.2f}"
)
ax.set_xlabel("Incident $H_s$ [m]")
ax.set_ylabel("MIZ width [km]")
ax.legend()
ax.grid(True)

ax = axes[3]
sns.regplot(
    data=df_reduced_plot,
    x=hs_col,
    y=miz_col,
    ax=ax,
    scatter_kws={"s": 18, "alpha": 0.25},
    line_kws={"linewidth": 2},
    label=f"Without unphysical, r={r_reduced:.2f}"
)

ax.set_xlabel("Incident $H_s$ [m]")
ax.set_ylabel("MIZ width [km]")
ax.legend()
ax.grid(True)

plt.show()

In [ ]:
miz_col = "miz_width_myedge_km"

miz = df_eff[miz_col].replace([np.inf, -np.inf], np.nan).dropna()

q1 = miz.quantile(0.25)
q3 = miz.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outside_mask = (miz < lower_bound) | (miz > upper_bound)

n_total = len(miz)
n_outside = outside_mask.sum()
frac_outside = n_outside / n_total

print(f"Q1 = {q1:.2f} km")
print(f"Q3 = {q3:.2f} km")
print(f"IQR = {iqr:.2f} km")
print(f"Lower bound = {lower_bound:.2f} km")
print(f"Upper bound = {upper_bound:.2f} km")
print(f"Points outside 1.5 IQR = {n_outside} / {n_total} ({100 * frac_outside:.2f}%)")

In [ ]:
# (df_all['hs_my_edge'] < 0.5).sum()

In [ ]:
# (df_all['hs_my_edge'] > 15).sum()

In [ ]:
hs_col = "hs_my_edge"
miz_col = "miz_width_myedge_km"

df_plot = df_eff[
    df_eff[hs_col].between(0.5, 10)
    & df_eff[miz_col].notna()
].copy()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

r = df_plot[hs_col].corr(df_plot[miz_col])

fig, ax = plt.subplots(figsize=(6, 4))

sns.regplot(
    data=df_plot,
    x=hs_col,
    y=miz_col,
    scatter_kws={"s": 12, "alpha": 0.3},
    line_kws={"color": "red", "linewidth": 2},
    ax=ax
)

ax.text(
    0.04,
    0.96,
    f"r = {r:.2f}\nn = {len(df_plot)}",
    transform=ax.transAxes,
    va="top",
    ha="left",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none")
)

ax.set_xlabel("Incident $H_s$ [m]")
ax.set_ylabel("MIZ width [km]")
ax.grid(True)

plt.show()

## Propagate waves along the AltiKa tracks
Using a similar method to CICE6-WIM, we propagate waves in 1D (assuming instantanous wave propagation; steady state conditions).
$$ S(x; \omega) = S(0; \omega)\,e^{-a_i\,\alpha(\omega)\,x}$$

Our method is described as follows
1. Use observed $H_s$ from AltiKa and the wave propagation method to estimate the $H_s$ corresponding with the observed inner-MIZ boundary.
2. We then use the same wave propagation method to calculate the "estimated MIZ width", using an observed $H_s$ and a $T_p=10.8\,$s (which is the average $T_p$ from WW3 outputs at the ice edge)
3. 



Assumptions:
- Bretschneider spectrum.
- $T_p$ is constant for time and space.
- The inner boundary of the MIZ corresponds with a constant $H_s$ value.
- Waves propagate directly along the tracks.

## $H_s$ at the interior MIZ boundary

### Read in aligned dataset

In [ ]:
from glob import glob
import pandas as pd

files = sorted(glob(
    "/g/data/ps29/nd0349/Fraser-2024/data/cleaned/"
    "*_altika_tracks_on_ww3_both_edges_n300_v0_15.csv"
))

if len(files) == 0:
    raise FileNotFoundError("No matching CSV files found")

df_aligned = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True,
)
# normalize columns
df_aligned = normalize_df(df_aligned)

df_aligned

In [ ]:
df_aligned.columns

In [ ]:
df_sample = df_aligned.copy()
df_sample = normalize_df(df_sample)

In [ ]:
df_sample["Tp_swell"] = estimate_tp_swell_from_hs(df_sample["hs_my_edge"])
df_sample["hs_inner_edge"] = df_sample.apply(
    lambda row: hs_after_distance(
        hs0=row["hs_my_edge"],
        aice=1.0,
        distance_km=row["mizWidthAlongTrackFromMyEdge"],
        tp=10,
    ),
    axis=1
)

In [ ]:
plt.scatter(df_sample["Tp_swell"], df_sample["hs_my_edge"])

In [ ]:
df_sample

In [ ]:
df_sample.columns

In [ ]:
df_sample = df_aligned.copy()
df_sample = normalize_df(df_sample)

In [ ]:
target_tp = df_sample["TP_edge_north_myedge"].replace([np.inf, -np.inf], np.nan).dropna().median()
target_mwd = df_sample["THM_edge_north_myedge"].replace([np.inf, -np.inf], np.nan).dropna().median()
df_sample["hs_inner_edge"] = df_sample.apply(
    lambda row: hs_after_distance(
        hs0=row["hs_my_edge"],
        distance_km=row["mizWidthAlongTrackFromMyEdge"],
        aice=row["ICE_track_mean_myedge"],
        tp=target_tp,
        mwd=target_mwd,
        direction_convention="from",
        use_southward_energy=True,
    ),
    axis=1
)

df_sample["hs_inner_edge_aice1"] = df_sample.apply(
    lambda row: hs_after_distance(
        hs0=row["hs_my_edge"],
        distance_km=row["mizWidthAlongTrackFromMyEdge"],
        aice=1.0,
        tp=target_tp,
        use_southward_energy=False,
    ),
    axis=1
)

In [ ]:
hs_col = "hs_my_edge"
hs_after_col = "hs_inner_edge"

x1 = df_sample[hs_col].replace([np.inf, -np.inf], np.nan).dropna()
x2 = df_sample[hs_after_col].replace([np.inf, -np.inf], np.nan).dropna()
x3 = df_sample["hs_inner_edge_aice1"].replace([np.inf, -np.inf], np.nan).dropna()

bins = np.linspace(
    min(x1.min(), x2.min()),
    max(x1.max(), x2.max()),
    60
)

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(
    x1,
    bins=bins,
    stat="density",
    kde=True,
    alpha=0.30,
    label="Incident $H_s$",
    ax=ax
)

sns.histplot(
    x2,
    bins=bins,
    stat="density",
    kde=True,
    alpha=0.30,
    label="$H_s$ after MIZ width",
    ax=ax
)


sns.histplot(
    x3,
    bins=bins,
    stat="density",
    kde=True,
    alpha=0.30,
    label="$H_s$ after MIZ width (aice=1.0)",
    ax=ax
)

ax.set_xlabel("$H_s$ [m]")
ax.set_ylabel("Density")
ax.legend()
ax.grid(True)

plt.xlim(0,10)

plt.show()

In [ ]:
median_incident = x1.median()
median_inner = x2.median()
median_inner2 = x3.median()

print(f"Median incident Hs = {median_incident:.2f} m")
print(f"Median Hs at inner edge = {median_inner:.2f} m")
print(f"Median Hs at inner edge = {median_inner2:.2f} m")

In [ ]:
df_sample["hs_fraction"] = df_sample["hs_inner_edge"]/df_sample["hs_my_edge"]


In [ ]:
hs_after_col = "swhFraction"

x = df_sample[hs_after_col].replace([np.inf, -np.inf], np.nan).dropna()

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(
    x,
    bins=50,
    stat="density",
    kde=True,
    alpha=0.35,
    ax=ax
)

ax.set_xlabel("$H_s$ fraction after attenuation [-]")
ax.set_ylabel("Density")
ax.grid(True)
# plt.xscale('log')
# plt.xlim(1e-3, 1)
plt.show()

In [ ]:
df_sample["swhFraction"].median()

In [ ]:
hs_after_col = "hs_inner_edge"

x = df_sample[hs_after_col].replace([np.inf, -np.inf], np.nan).dropna()

fig, ax = plt.subplots(figsize=(7, 4))

sns.histplot(
    x,
    bins=100,
    stat="density",
    kde=True,
    alpha=0.35,
    ax=ax
)

ax.set_xlabel("$H_s$ at inner edge [m]")
ax.set_ylabel("Density")
ax.grid(True)
plt.xlim(0,5)
plt.show()

In [ ]:
df_sample["hs_inner_edge"].median()

In [ ]:
df_sample["distance_until_hs_fraction_km"] = df_sample["hs_my_edge"].apply(
    lambda hs0: distance_until_hs_fraction_below(
        hs0=hs0,
        tp=10,
        aice=df_sample["ice_track_mean_myedge"].median(),
        freqs=freqs,
        fraction_threshold=df_sample["hs_fraction"].median(),
        alpha_units="m-1",
    )
)

In [ ]:
df_sample["hs_inner_edge"].replace([np.inf, -np.inf], np.nan).dropna().median()

In [ ]:
# df_sample['mwd'].values

In [ ]:
# southward_energy_fraction_cos2(-3, direction_convention="from", n_dirs=720)

In [ ]:
target_hs = df_sample["hs_inner_edge"].replace([np.inf, -np.inf], np.nan).dropna().median()
target_hs2 = df_sample["hs_inner_edge_aice1"].replace([np.inf, -np.inf], np.nan).dropna().median()
target_tp = df_sample["TP_edge_north_myedge"].replace([np.inf, -np.inf], np.nan).dropna().median()

df_sample["est_miz_width_altika_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["hs_my_edge"],
        target_hs=target_hs,
        tp=target_tp,
        aice=row["ICE_track_mean_myedge"],
        # aice=row["mean_siconc_myedge_track"],
        # tp=row["tp"],
    ),
    axis=1
)


df_sample["est_miz_width_altika_eff_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["hs_my_edge"],
        target_hs=target_hs,
        tp=target_tp,
        aice=1.0,
        # tp=row["tp"],
    ),
    axis=1
)

In [ ]:
df_sample["est_miz_width_altika_aice1_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["hs_my_edge"],
        target_hs=target_hs2,
        tp=target_tp,
        aice=1.0,
        # aice=row["mean_siconc_myedge_track"],
        # tp=row["tp"],
    ),
    axis=1
)

df_sample["est_miz_width_altika_aice1_eff_km"] = df_sample["est_miz_width_altika_km"] * df_sample["ICE_track_mean_myedge"]


In [ ]:
df_sample["est_miz_width_ww3_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["HS_edge_north_myedge"],
        target_hs=target_hs,
        tp=target_tp,
        aice=row["ICE_track_mean_myedge"],
        # tp=row["tp"],
    ),
    axis=1
)

# df_sample["est_miz_width_ww3_eff_km"] = df_sample["est_miz_width_ww3_km"] * df_sample["ICE_track_mean_myedge"]

df_sample["est_miz_width_ww3_eff_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["HS_edge_north_myedge"],
        target_hs=target_hs,
        tp=target_tp,
    ),
    axis=1
)

In [ ]:
df_sample["est_miz_width_ww3_tp_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["HS_edge_north_myedge"],
        target_hs=target_hs,
        tp=row["TP_edge_north_myedge"],
        aice=row["ICE_track_mean_myedge"],
    ),
    axis=1
)

# df_sample["est_miz_width_ww3_tp_eff_km"] = df_sample["est_miz_width_ww3_tp_km"] * df_sample["ICE_track_mean_myedge"]
df_sample["est_miz_width_ww3_tp_eff_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["HS_edge_north_myedge"],
        target_hs=target_hs,
        tp=row["TP_edge_north_myedge"],
        aice=1.0,
    ),
    axis=1
)

In [ ]:
df_sample["est_miz_width_ww3_south_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["HS_edge_north_myedge"],
        target_hs=target_hs,
        tp=row["TP_edge_north_myedge"],
        mwd=row["THM_edge_north_myedge"],  # radians
        aice=row["ICE_track_mean_myedge"],
        use_southward_energy=True,
        direction_convention="from",
    ),
    axis=1,
)

# df_sample["est_miz_width_ww3_south_eff_km"] = df_sample["est_miz_width_ww3_south_km"] * df_sample["ICE_track_mean_myedge"]

df_sample["est_miz_width_ww3_south_eff_km"] = df_sample.apply(
    lambda row: distance_until_hs_below_target(
        hs0=row["HS_edge_north_myedge"],
        target_hs=target_hs,
        aice=1.0,
        tp=row["TP_edge_north_myedge"],
        mwd=row["THM_edge_north_myedge"],  # radians
        use_southward_energy=True,
        direction_convention="from",
    ),
    axis=1,
)

In [ ]:
df_sample[[
    "hs_my_edge",
    "mizWidthAlongTrackFromMyEdge",
    "distance_until_swh_fraction_km",
    "est_miz_width_altika_km",
    "est_miz_width_ww3_km",
    "est_miz_width_altika_eff_km",
    "est_miz_width_ww3_eff_km"
]].describe()

In [ ]:
corr_cols = [
    "hs_my_edge",
    "mizWidthAlongTrackFromMyEdge",
    "distance_until_swh_fraction_km",
    "est_miz_width_altika_km",
    "est_miz_width_altika_aice1_km",
    "est_miz_width_ww3_km",
    "est_miz_width_altika_eff_km",
    "est_miz_width_ww3_eff_km",
]

corr_matrix = (
    df_sample[corr_cols]
    .replace([np.inf, -np.inf], np.nan)
    .corr()
)

corr_matrix

In [ ]:
from scipy.stats import wasserstein_distance

max_x = 1000

cols = [
    "mizWidthAlongTrackFromMyEdge",
    # "distance_until_swh_fraction_008_km",
    "est_miz_width_altika_km",
    "est_miz_width_ww3_km",
]

labels = {
    "mizWidthAlongTrackFromMyEdge": "Observed MIZ width",
    # "distance_until_swh_fraction_008_km": "Distance to $H_s/H_{s0}=0.08$",
    "est_miz_width_altika_km": "Distance to median inner-edge $H_s$",
    "est_miz_width_ww3_km": "Distance to median inner-edge $H_s$ (WW3)",
}

plot_data = {}
for col in cols:
    x = df_sample[col].replace([np.inf, -np.inf], np.nan).dropna()
    x = x[(x >= 0) & (x <= max_x)]
    plot_data[col] = x

In [ ]:
def distribution_comparison(x, y, bins):
    wd = wasserstein_distance(x, y)

    px, _ = np.histogram(x, bins=bins, density=True)
    py, _ = np.histogram(y, bins=bins, density=True)

    bin_widths = np.diff(bins)
    overlap = np.sum(np.minimum(px, py) * bin_widths)

    return wd, overlap

In [ ]:
bins = np.linspace(0, max_x, 60)

obs_col = "mizWidthAlongTrackFromMyEdge"

comparisons = []

for col in cols:
    if col == obs_col:
        continue

    wd, overlap = distribution_comparison(
        plot_data[obs_col],
        plot_data[col],
        bins=bins
    )

    comparisons.append({
        "comparison": f"{labels[obs_col]} vs {labels[col]}",
        "wasserstein_km": wd,
        "overlap": overlap,
        "n_obs": len(plot_data[obs_col]),
        "n_other": len(plot_data[col]),
    })

comparison_df = pd.DataFrame(comparisons)

print(comparison_df)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for col in cols:
    x = plot_data[col]

    sns.histplot(
        x,
        bins=bins,
        stat="density",
        kde=True,
        alpha=0.25,
        label=f"{labels[col]}, n={len(x)}",
        ax=ax
    )

stats_text = "\n".join(
    f"{row['comparison'].split(' vs ')[1]}:\n"
    f"  W = {row['wasserstein_km']:.1f} km, overlap = {row['overlap']:.2f}"
    for _, row in comparison_df.iterrows()
)

ax.text(
    0.98,
    0.95,
    stats_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(facecolor="white", alpha=0.85, edgecolor="none")
)

ax.set_xlim(0, max_x)
ax.set_xlabel("Distance / width [km]")
ax.set_ylabel("Density")
ax.legend()
ax.grid(True)

plt.show()

In [ ]:
obs_col = "mizWidthAlongTrackFromMyEdge"
inner_col = "est_miz_width_altika_km"

df_scatter = df_sample[[obs_col, inner_col]].replace([np.inf, -np.inf], np.nan).dropna()

# Optional: keep same max range as before
df_scatter = df_scatter[
    (df_scatter[obs_col] >= 0) & (df_scatter[obs_col] <= 1000) &
    (df_scatter[inner_col] >= 0) & (df_scatter[inner_col] <= 1000)
]

r = df_scatter[obs_col].corr(df_scatter[inner_col])

fig, ax = plt.subplots(figsize=(5.5, 5))

sns.regplot(
    data=df_scatter,
    x=obs_col,
    y=inner_col,
    scatter_kws={"s": 14, "alpha": 0.3},
    line_kws={"color": "red", "linewidth": 2},
    ax=ax
)

lims = [
    min(df_scatter[obs_col].min(), df_scatter[inner_col].min()),
    max(df_scatter[obs_col].max(), df_scatter[inner_col].max()),
]

ax.plot(lims, lims, "k--", linewidth=1, label="1:1")

ax.text(
    0.04,
    0.96,
    f"r = {r:.2f}\nn = {len(df_scatter)}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    bbox=dict(facecolor="white", alpha=0.85, edgecolor="none")
)

ax.set_xlabel("Observed MIZ width [km]")
ax.set_ylabel("Distance to median inner-edge $H_s$ [km]")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
ax.grid(True)

plt.show()

In [ ]:
df_sample

### Average temporally and spatially

In [ ]:
df_sample.columns

In [ ]:
# create effective width columns using the available column names
era5_eff_name = choose_col(df_sample, 'mizWidthAlongTrackFromMyEdge_eff')
obs_miz_col = choose_col(df_sample, 'mizWidthAlongTrackFromMyEdge')
ice_mean_col = choose_col(df_sample, 'ICE_track_mean_myedge')
df_sample[era5_eff_name] = df_sample[obs_miz_col] * df_sample[ice_mean_col]
df_sample["ww3_miz_width_hs05_km_myedge_eff"] = df_sample[choose_col(df_sample, 'ww3_miz_width_hs05_km_myedge')] * df_sample[ice_mean_col]
# normalize working time/lon columns
df_sample['time'] = pd.to_datetime(df_sample[choose_col(df_sample, 'first_meas_time')])
df_sample['lon'] = df_sample[choose_col(df_sample, 'lonAtMyEdge')]

In [ ]:
# Helper to pick the existing column name (prefer canonical rename if present)
def choose_col(df, old_name):
    new_name = rename_map.get(old_name, old_name)
    return new_name if new_name in df.columns else old_name

# pick the actual column names from df_sample (robust to whether df was normalized)
time_col = choose_col(df_sample, 'first_meas_time')
lon_col = choose_col(df_sample, 'lonAtMyEdge')
obs_col = choose_col(df_sample, 'mizWidthAlongTrackFromMyEdge')
inner_col = choose_col(df_sample, 'est_miz_width_altika_km')
inner_model_col = choose_col(df_sample, 'est_miz_width_ww3_km')
era5_eff_col = choose_col(df_sample, 'mizWidthAlongTrackFromMyEdge_eff')
inner_col_eff = choose_col(df_sample, 'est_miz_width_altika_eff_km')
inner_col_model = choose_col(df_sample, 'est_miz_width_ww3_km')
inner_col_model_eff = choose_col(df_sample, 'est_miz_width_ww3_eff_km')

# other model/metadata columns (use choose_col to be robust)
cols_extra = [
    'est_miz_width_altika_aice1_km',
    'est_miz_width_altika_aice1_eff_km',
    'est_miz_width_ww3_tp_km',
    'est_miz_width_ww3_tp_eff_km',
    'est_miz_width_ww3_south_km',
    'est_miz_width_ww3_south_eff_km',
    'ICE_track_mean_myedge',
    'TP_edge_myedge',
    'HS_edge_myedge',
    'TP_edge_north_myedge',
    'HS_edge_north_myedge',
    'hs_my_edge',
    'ww3_miz_width_hs05_km_myedge',
    'ww3_miz_width_hs05_km_myedge_eff',
]
value_cols = [obs_col, inner_col, inner_col_eff, inner_model_col, inner_col_model_eff, era5_eff_col] + [choose_col(df_sample, c) for c in cols_extra]

# build df_line using the chosen time/lon columns and the value columns
df_line = (
    df_sample[[time_col, lon_col] + value_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

# normalize the working time/longitude columns for downstream code
df_line['time'] = pd.to_datetime(df_line[time_col])
df_line['date'] = df_line['time'].dt.floor('D')
df_line['lon_bin'] = np.floor(df_line[lon_col] % 360).astype(int)

df_by_date = (
    df_line
    .groupby('date', as_index=False)[value_cols]
    .mean()
)

df_by_lon = (
    df_line
    .groupby('lon_bin', as_index=False)[value_cols]
    .mean()
)

# inner_col_eff = "est_miz_width_altika_eff_km"
# df_by_date[inner_col_eff] = df_by_date["est_miz_width_altika_km"] * df_extra_by_date["mean_siconc_myedge_track"]
# df_by_lon[inner_col_eff] = (
#     df_by_lon["est_miz_width_altika_km"] * df_extra_by_lon["mean_siconc_myedge_track"]
# )

# inner_col_model_eff = "est_miz_width_ww3_eff_km"
# df_by_date[inner_col_model_eff] = df_by_date["est_miz_width_ww3_km"] * df_extra_by_date["mean_siconc_myedge_track"]
# df_by_lon[inner_col_model_eff] = (
#     df_by_lon["est_miz_width_ww3_km"] * df_extra_by_lon["mean_siconc_myedge_track"]
# )

In [ ]:
df_by_date.head()

In [ ]:
df_by_lon.head()

In [ ]:
corr_cols = [
    inner_col,
    inner_col_eff,
    inner_model_col,
    inner_col_model_eff,
    "est_miz_width_altika_aice1_km",
    "est_miz_width_altika_aice1_eff_km",
    "est_miz_width_ww3_tp_km",
    "est_miz_width_ww3_tp_eff_km",
    "est_miz_width_ww3_south_km",
    "est_miz_width_ww3_south_eff_km",
    "ww3_miz_width_hs05_km_myedge",
    "ww3_miz_width_hs05_km_myedge_eff",
]

corr_group_labels = {
    inner_col: "AltiKa",
    inner_col_eff: "AltiKa",

    "est_miz_width_altika_aice1_km": "AltiKa (aice=1)",
    "est_miz_width_altika_aice1_eff_km": "AltiKa (aice=1)",

    inner_model_col: "WW3",
    inner_col_model_eff: "WW3",

    "est_miz_width_ww3_tp_km": "WW3 $T_p$",
    "est_miz_width_ww3_tp_eff_km": "WW3 $T_p$",

    "est_miz_width_ww3_south_km": "WW3 $T_p$, $\\theta$",
    "est_miz_width_ww3_south_eff_km": "WW3 $T_p$, $\\theta$",

    "ww3_miz_width_hs05_km_myedge": "WW3 MIZ width",
    "ww3_miz_width_hs05_km_myedge_eff": "WW3 MIZ width",
}

corr_colors = {
    # AltiKa estimate
    inner_col: "tab:blue",
    inner_col_eff: "tab:blue",

    "est_miz_width_altika_aice1_km": "tab:blue",
    "est_miz_width_altika_aice1_eff_km": "tab:blue",

    # WW3 estimate
    inner_model_col: "tab:purple",
    inner_col_model_eff: "tab:purple",

    # WW3 estimate with Tp
    "est_miz_width_ww3_tp_km": "tab:orange",
    "est_miz_width_ww3_tp_eff_km": "tab:orange",

    # WW3 estimate with Tp + direction
    "est_miz_width_ww3_south_km": "tab:red",
    "est_miz_width_ww3_south_eff_km": "tab:red",

    # WW3 MIZ width
    "ww3_miz_width_hs05_km_myedge": "tab:green",
    "ww3_miz_width_hs05_km_myedge_eff": "tab:green",
}

def valid_pair(df, xcol, ycol):
    return (
        df[[xcol, ycol]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )


def pair_correlation(df, xcol, ycol):
    df_valid = valid_pair(df, xcol, ycol)

    if len(df_valid) < 2:
        return np.nan, len(df_valid)

    return df_valid[xcol].corr(df_valid[ycol]), len(df_valid)


def relationship_stats(df, xcol, ycol):
    df_valid = valid_pair(df, xcol, ycol)

    if len(df_valid) < 2:
        return np.nan, np.nan, np.nan, len(df_valid)

    r = df_valid[xcol].corr(df_valid[ycol])
    rmse = np.sqrt(np.mean((df_valid[xcol] - df_valid[ycol]) ** 2))
    bias = np.mean(df_valid[ycol] - df_valid[xcol])

    return r, rmse, bias, len(df_valid)


def corr_against_obs(df, obs_col, cols, width_kind):
    rows = []

    for col in cols:
        r, rmse, bias, n = relationship_stats(df, obs_col, col)

        rows.append({
            "col": col,
            "label": corr_group_labels.get(col, col),
            "width_kind": width_kind,
            "color": corr_colors.get(col, "0.35"),
            "r": r,
            "r2": r ** 2,
            "rmse": rmse,
            "bias": bias,
            "n": n,
        })

    return pd.DataFrame(rows)



def make_stats_text(df, comparisons):
    lines = []

    for name, cols in comparisons.items():
        r, rmse, bias, n = relationship_stats(
            df,
            cols["obs"],
            cols["model"],
        )

        lines.append(
            f"{name}\n"
            f"$r$ = {r:.2f}, RMSE = {rmse:.1f} km\n"
            f"Bias = {bias:.1f} km, $n$ = {n}"
        )

    return "\n\n".join(lines)

import matplotlib.colors as mcolors

def lighten_color(color, amount=0.55):
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)

In [ ]:
corr_abs_cols = [
    inner_col,
    "est_miz_width_altika_aice1_km",
    inner_model_col,
    "est_miz_width_ww3_tp_km",
    "est_miz_width_ww3_south_km",
    "ww3_miz_width_hs05_km_myedge",
]

corr_eff_cols = [
    inner_col_eff,
    "est_miz_width_altika_aice1_eff_km",
    inner_col_model_eff,
    "est_miz_width_ww3_tp_eff_km",
    "est_miz_width_ww3_south_eff_km",
    "ww3_miz_width_hs05_km_myedge_eff",
]

corr_date = pd.concat(
    [
        corr_against_obs(df_by_date, obs_col, corr_abs_cols, "Absolute"),
        corr_against_obs(df_by_date, era5_eff_col, corr_eff_cols, "Effective"),
    ],
    ignore_index=True,
)

corr_lon = pd.concat(
    [
        corr_against_obs(df_by_lon, obs_col, corr_abs_cols, "Absolute"),
        corr_against_obs(df_by_lon, era5_eff_col, corr_eff_cols, "Effective"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(4, 6),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

bar_width = 0.36

for ax, corr_df, title in zip(
    axes,
    [corr_date, corr_lon],
    ["Averaged by date", "Averaged by longitude"],
):
    labels = corr_df["label"].drop_duplicates().to_list()
    x = np.arange(len(labels))

    abs_df = (
        corr_df[corr_df["width_kind"] == "Absolute"]
        .set_index("label")
        .reindex(labels)
    )

    eff_df = (
        corr_df[corr_df["width_kind"] == "Effective"]
        .set_index("label")
        .reindex(labels)
    )

    base_colors = abs_df["color"].to_list()
    eff_colors = [lighten_color(c, amount=0.45) for c in base_colors]

    ax.bar(
        x - bar_width / 2,
        abs_df["r"],
        width=bar_width,
        color=base_colors,
        label="Absolute",
    )

    ax.bar(
        x + bar_width / 2,
        eff_df["r"],
        width=bar_width,
        color=eff_colors,
        edgecolor=base_colors,
        # hatch="//",
        label="Effective",
    )

    ax.axhline(0, color="0.4", linewidth=1)
    ax.set_title(title)
    ax.set_ylabel("Correlation")
    ax.set_ylim(0, 1)
    # ax.legend(title="")

axes[-1].set_xticks(np.arange(len(labels)))
axes[-1].set_xticklabels(labels, rotation=45, ha="right")

plt.show()

In [ ]:
comparisons = {
    "Altika absolute": {"obs": obs_col, "model": inner_col},
    "WW3 absolute": {"obs": obs_col, "model": inner_col_model},
    "Altika effective": {"obs": era5_eff_col, "model": inner_col_eff},
    "WW3 effective": {"obs": era5_eff_col, "model": inner_col_model_eff},
}

date_stats_text = make_stats_text(df_by_date, comparisons)
lon_stats_text = make_stats_text(df_by_lon, comparisons)

In [ ]:
df_line = (
    df_sample[["time", "lon"] + value_cols]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

In [ ]:
def plot_width_comparison(
    df_by_date,
    df_by_lon,
    obs_abs_col,
    obs_eff_col,
    model_abs_col,
    model_eff_col,
    title,
    model_abs_label,
    model_eff_label,
    model_color,
    model_eff_color,
):
    comparisons = {
        f"{title} absolute": {
            "obs": obs_abs_col,
            "model": model_abs_col,
        },
        f"{title} effective": {
            "obs": obs_eff_col,
            "model": model_eff_col,
        },
    }

    date_stats_text = make_stats_text(df_by_date, comparisons)
    lon_stats_text = make_stats_text(df_by_lon, comparisons)

    labels = {
        obs_abs_col: "Observed MIZ width (AltiKa)",
        obs_eff_col: "Observed effective MIZ width (AltiKa on ERA5 ice)",
        model_abs_col: model_abs_label,
        model_eff_col: model_eff_label,
    }

    colors = {
        obs_abs_col: "black",
        obs_eff_col: "0.55",
        model_abs_col: model_color,
        model_eff_col: model_eff_color,
    }

    plot_order = [
        obs_abs_col,
        model_abs_col,
        obs_eff_col,
        model_eff_col,
    ]

    fig, axes = plt.subplots(
        2, 1,
        figsize=(11, 8),
        constrained_layout=True,
    )

    # ---------------- Date averages ----------------
    ax = axes[0]

    for col in plot_order:
        ax.plot(
            df_by_date["date"],
            df_by_date[col],
            color=colors[col],
            linewidth=1.8,
            label=labels[col],
        )

    ax.text(
        0.02,
        0.95,
        date_stats_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
    )

    # ax.set_title(f"{title}: date averages")
    ax.set_xlabel("Date")
    ax.set_ylabel("Mean distance / width [km]")
    ax.set_ylim(0, 400)
    ax.legend(loc="upper right", fontsize=8)

    # ---------------- Longitude averages ----------------
    ax = axes[1]

    for col in plot_order:
        ax.plot(
            df_by_lon["lon_bin"],
            df_by_lon[col],
            color=colors[col],
            linewidth=1.8,
            label=labels[col],
        )

    ax.text(
        0.02,
        0.95,
        lon_stats_text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=8,
        bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
    )

    # ax.set_title(f"{title}: longitude averages")
    ax.set_xlabel("Longitude [$^\circ$E]")
    ax.set_ylabel("Mean distance / width [km]")
    ax.set_xlim(0, 359)
    ax.set_ylim(0, 400)
    ax.legend(loc="upper right", fontsize=8)

    return fig, axes

In [ ]:
fig_altika, axes_altika = plot_width_comparison(
    df_by_date=df_by_date,
    df_by_lon=df_by_lon,
    obs_abs_col=obs_col,
    obs_eff_col=era5_eff_col,
    model_abs_col=inner_col,
    model_eff_col=inner_col_eff,
    title="Altika estimate",
    model_abs_label=f"AltiKa: MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_eff_label=f"AltiKa: Effective MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_color="tab:blue",
    model_eff_color="lightskyblue",
)

plt.show()

In [ ]:
fig_altika, axes_altika = plot_width_comparison(
    df_by_date=df_by_date,
    df_by_lon=df_by_lon,
    obs_abs_col=obs_col,
    obs_eff_col=era5_eff_col,
    model_abs_col="est_miz_width_altika_aice1_km",
    model_eff_col="est_miz_width_altika_aice1_eff_km",
    title="Altika estimate",
    model_abs_label=f"AltiKa: MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_eff_label=f"AltiKa: Effective MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_color="tab:blue",
    model_eff_color="lightskyblue",
)

plt.show()

In [ ]:
fig_ww3, axes_ww3 = plot_width_comparison(
    df_by_date=df_by_date,
    df_by_lon=df_by_lon,
    obs_abs_col=obs_col,
    obs_eff_col=era5_eff_col,
    model_abs_col=inner_col_model,
    model_eff_col=inner_col_model_eff,
    title="WW3 estimate",
    model_abs_label=f"WW3: MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_eff_label=f"WW3: Effective MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_color="tab:purple",
    model_eff_color="plum",
)

plt.show()

In [ ]:
fig_ww3, axes_ww3 = plot_width_comparison(
    df_by_date=df_by_date,
    df_by_lon=df_by_lon,
    obs_abs_col=obs_col,
    obs_eff_col=era5_eff_col,
    model_abs_col="est_miz_width_ww3_tp_km",
    model_eff_col="est_miz_width_ww3_tp_eff_km",
    title="WW3 estimate",
    model_abs_label=f"WW3: MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_eff_label=f"WW3: Effective MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_color="tab:purple",
    model_eff_color="plum",
)

plt.show()

In [ ]:
fig_ww3, axes_ww3 = plot_width_comparison(
    df_by_date=df_by_date,
    df_by_lon=df_by_lon,
    obs_abs_col=obs_col,
    obs_eff_col=era5_eff_col,
    model_abs_col="est_miz_width_ww3_south_km",
    model_eff_col="est_miz_width_ww3_south_eff_km",
    title="WW3 estimate",
    model_abs_label=f"WW3: MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_eff_label=f"WW3: Effective MIZ estimate ($H_s$ > {target_hs:.2f} m)",
    model_color="tab:purple",
    model_eff_color="plum",
)

plt.show()

In [ ]:
fig_mizwidth, axes_mizwidth = plot_width_comparison(
    df_by_date=df_by_date,
    df_by_lon=df_by_lon,
    obs_abs_col=obs_col,
    obs_eff_col=era5_eff_col,
    model_abs_col="ww3_miz_width_hs05_km_myedge",
    model_eff_col="ww3_miz_width_hs05_km_myedge_eff",
    title="WW3 MIZ width",
    model_abs_label="WW3 MIZ width ($H_s>$0.5 m)",
    model_eff_label="WW3 effective MIZ width ($H_s>$0.5 m)",
    model_color="tab:green",
    model_eff_color="lightgreen",
)
plt.show()

In [ ]:
hist_colors = {
    # Observed / AltiKa observations
    obs_col: "black",
    era5_eff_col: "black",
    "hs_my_edge": "black",

    # WW3 wave height
    "HS_edge_north_myedge": "tab:purple",

    # AltiKa estimates
    inner_col: "tab:blue",
    inner_col_eff: "tab:blue",

    # WW3 Hs-based estimate
    inner_col_model: "tab:purple",
    inner_col_model_eff: "tab:purple",

    # WW3 Tp-based estimate
    "est_miz_width_ww3_tp_km": "tab:orange",
    "est_miz_width_ww3_tp_eff_km": "tab:orange",

    # WW3 Tp + direction estimate
    "est_miz_width_ww3_south_km": "tab:red",
    "est_miz_width_ww3_south_eff_km": "tab:red",

    # WW3 MIZ-width product
    "ww3_miz_width_hs05_km_myedge": "tab:green",
    "ww3_miz_width_hs05_km_myedge_eff": "tab:green",
}

hist_labels = {
    # Wave heights
    "hs_my_edge": "AltiKa $H_s$",
    "HS_edge_north_myedge": "WW3 $H_s$",

    # Absolute widths
    obs_col: "Observed",
    inner_col: "AltiKa estimate",
    inner_col_model: "WW3 $H_s$ estimate",
    "est_miz_width_ww3_tp_km": "WW3 $T_p$ estimate",
    "est_miz_width_ww3_south_km": "WW3 $T_p, \\theta$ estimate",
    "ww3_miz_width_hs05_km_myedge": "WW3 MIZ width",

    # Effective widths
    era5_eff_col: "Observed effective",
    inner_col_eff: "AltiKa effective estimate",
    inner_col_model_eff: "WW3 $H_s$ effective estimate",
    "est_miz_width_ww3_tp_eff_km": "WW3 $T_p$ effective estimate",
    "est_miz_width_ww3_south_eff_km": "WW3 $T_p, \\theta$ effective estimate",
    "ww3_miz_width_hs05_km_myedge_eff": "WW3 effective MIZ width",
}

wave_cols = [
    "hs_my_edge",
    "HS_edge_north_myedge",
]

abs_width_cols = [
    obs_col,
    inner_col,
    inner_col_model,
    "est_miz_width_ww3_tp_km",
    "est_miz_width_ww3_south_km",
    "ww3_miz_width_hs05_km_myedge",
]

eff_width_cols = [
    era5_eff_col,
    inner_col_eff,
    inner_col_model_eff,
    "est_miz_width_ww3_tp_eff_km",
    "est_miz_width_ww3_south_eff_km",
    "ww3_miz_width_hs05_km_myedge_eff",
]

fig, axes = plt.subplots(
    1,
    3,
    figsize=(19, 4.5),
    constrained_layout=True,
)

plot_groups = [
    {
        "ax": axes[0],
        "cols": wave_cols,
        "title": "Significant wave height",
        "xlabel": "$H_s$ [m]",
        "xlim": None,
    },
    {
        "ax": axes[1],
        "cols": abs_width_cols,
        "title": "Absolute MIZ width",
        "xlabel": "MIZ width [km]",
        "xlim": (0, 400),
    },
    {
        "ax": axes[2],
        "cols": eff_width_cols,
        "title": "Effective MIZ width",
        "xlabel": "Effective MIZ width [km]",
        "xlim": (0, 400),
    },
]

df_sample["est_miz_width_altika_eff_km"] = (
    df_sample["est_miz_width_altika_km"]
    * df_sample["ICE_track_mean_myedge"]
)

df_sample["est_miz_width_ww3_eff_km"] = (
    df_sample["est_miz_width_ww3_km"]
    * df_sample["ICE_track_mean_myedge"]
)

show_histograms = False  # True = histogram + KDE, False = KDE only

for group in plot_groups:
    ax = group["ax"]

    for col in group["cols"]:
        data = (
            df_sample[col]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if show_histograms:
            sns.histplot(
                data,
                ax=ax,
                stat="density",
                bins=40,
                element="step",
                fill=False,
                linewidth=1.4,
                color=hist_colors[col],
                label=hist_labels[col],
            )

            kde_label = None
        else:
            kde_label = hist_labels[col]

        sns.kdeplot(
            data,
            ax=ax,
            color=hist_colors[col],
            linewidth=2,
            label=kde_label,
        )

    ax.set_title(group["title"])
    ax.set_xlabel(group["xlabel"])
    ax.set_ylabel("Density")

    if group["xlim"] is not None:
        ax.set_xlim(*group["xlim"])

    ax.legend(fontsize=8)

plt.show()

In [ ]:
# df_sample.columns

In [ ]:
# inner_col_eff

In [ ]:
effective_scatter_comparisons = [
    {
        "title": "AltiKa effective",
        "model_col": inner_col_eff,
        "color": "tab:blue",
        "label": f"AltiKa effective estimate ($H_s$ > {target_hs:.2f} m)",
    },
    {
        "title": "WW3 effective estimate",
        "model_col": inner_col_model_eff,
        "color": "tab:purple",
        "label": f"WW3 effective estimate ($H_s$ > {target_hs:.2f} m)",
    },
    {
        "title": "WW3 effective estimate ($T_p$)",
        "model_col": "est_miz_width_ww3_tp_eff_km",
        "color": "tab:orange",
        "label": f"WW3 effective estimate ($H_s$ > {target_hs:.2f} m)",
    },
    {
        "title": "WW3 effective estimate ($T_p, \\theta$)",
        "model_col": "est_miz_width_ww3_south_eff_km",
        "color": "tab:red",
        "label": f"WW3 effective estimate ($H_s$ > {target_hs:.2f} m)",
    },
    {
        "title": "WW3 effective MIZ width",
        "model_col": "ww3_miz_width_hs05_km_myedge_eff",
        "color": "tab:green",
        "label": "WW3 effective MIZ width ($H_s$ > 0.5 m)",
    },
]

fig, axes = plt.subplots(
    2, len(effective_scatter_comparisons),
    figsize=(4*len(effective_scatter_comparisons), 8),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

datasets = [
    {
        "df": df_by_date,
        "row_label": "Date averages",
    },
    {
        "df": df_by_lon,
        "row_label": "Longitude averages",
    },
]

for row, dataset in enumerate(datasets):
    df_plot = dataset["df"]

    for col_idx, comp in enumerate(effective_scatter_comparisons):
        ax = axes[row, col_idx]

        xcol = comp["model_col"]
        ycol = era5_eff_col

        df_valid = (
            df_plot[[xcol, ycol]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        ax.scatter(
            df_valid[xcol],
            df_valid[ycol],
            s=18,
            alpha=0.7,
            color=comp["color"],
            edgecolor="none",
        )

        xy_max = np.nanmax([
            df_valid[xcol].max(),
            df_valid[ycol].max(),
            350,
        ])

        ax.plot(
            [0, xy_max],
            [0, xy_max],
            color="0.4",
            linewidth=1,
            linestyle="--",
        )

        r, rmse, bias, n = relationship_stats(df_plot, ycol, xcol)

        ax.text(
            0.05,
            0.95,
            f"$r$ = {r:.2f}\nRMSE = {rmse:.1f} km\nBias = {bias:.1f} km\nn = {n}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        if row == 0:
            ax.set_title(comp["title"])

        if col_idx == 0:
            ax.set_ylabel(f"{dataset['row_label']}\nObserved effective MIZ width [km]")
        else:
            ax.set_ylabel("")

        if row == 1:
            ax.set_xlabel("Model effective MIZ width [km]")
        else:
            ax.set_xlabel("")

        ax.set_xlim(0, 350)
        ax.set_ylim(0, 350)
        ax.set_aspect("equal", adjustable="box")

plt.show()

In [ ]:
miz_hs_comparisons = [
    {
        "title": "AltiKa observed",
        "abs_width_col": obs_col,
        "eff_width_col": era5_eff_col,
        "hs_col": "hs_my_edge",
        "abs_label": "Absolute",
        "eff_label": "Effective",
        "color": "black",
        "xlabel": "AltiKa $H_s$ [m]",
    },
    {
        "title": "AltiKa estimate",
        "abs_width_col": inner_col,
        "eff_width_col": inner_col_eff,
        "hs_col": "hs_my_edge",
        "abs_label": "Absolute",
        "eff_label": "Effective",
        "color": "tab:blue",
        "xlabel": "AltiKa $H_s$ [m]",
    },
    {
        "title": "WW3 estimate",
        "abs_width_col": inner_col_model,
        "eff_width_col": inner_col_model_eff,
        "hs_col": "HS_edge_north_myedge",
        "abs_label": "Absolute",
        "eff_label": "Effective",
        "color": "tab:purple",
        "xlabel": "WW3 $H_s$ [m]",
    },
    {
        "title": "WW3 estimate ($T_p$)",
        "abs_width_col": "est_miz_width_ww3_tp_km",
        "eff_width_col": "est_miz_width_ww3_tp_eff_km",
        "hs_col": "HS_edge_north_myedge",
        "abs_label": "Absolute",
        "eff_label": "Effective",
        "color": "tab:orange",
        "xlabel": "WW3 $H_s$ [m]",
    },
    {
        "title": "WW3 estimate ($T_p, \\theta$)",
        "abs_width_col": "est_miz_width_ww3_south_km",
        "eff_width_col": "est_miz_width_ww3_south_eff_km",
        "hs_col": "HS_edge_north_myedge",
        "abs_label": "Absolute",
        "eff_label": "Effective",
        "color": "tab:red",
        "xlabel": "WW3 $H_s$ [m]",
    },
    
    {
        "title": "WW3 MIZ width",
        "abs_width_col": "ww3_miz_width_hs05_km_myedge",
        "eff_width_col": "ww3_miz_width_hs05_km_myedge_eff",
        "hs_col": "HS_edge_north_myedge",
        "abs_label": "Absolute",
        "eff_label": "Effective",
        "color": "tab:green",
        "xlabel": "WW3 $H_s$ [m]",
    },
]

fig, axes = plt.subplots(
    2,
    len(miz_hs_comparisons),
    figsize=(4*len(miz_hs_comparisons), 8),
    constrained_layout=True,
    sharey=True,
    sharex="row",
)

datasets = [
    {
        "df": df_by_date,
        "row_label": "Date averages",
    },
    {
        "df": df_by_lon,
        "row_label": "Longitude averages",
    },
]

for row, dataset in enumerate(datasets):
    df_plot = dataset["df"]

    for col_idx, comp in enumerate(miz_hs_comparisons):
        ax = axes[row, col_idx]

        xcol = comp["hs_col"]
        abs_col = comp["abs_width_col"]
        eff_col = comp["eff_width_col"]

        df_abs = (
            df_plot[[xcol, abs_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        df_eff = (
            df_plot[[xcol, eff_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        ax.scatter(
            df_abs[xcol],
            df_abs[abs_col],
            s=18,
            alpha=0.45,
            color=comp["color"],
            marker="o",
            edgecolor="none",
            label=comp["abs_label"],
        )

        ax.scatter(
            df_eff[xcol],
            df_eff[eff_col],
            s=18,
            alpha=0.75,
            color=comp["color"],
            marker="^",
            edgecolor="none",
            label=comp["eff_label"],
        )

        r_abs = df_abs[xcol].corr(df_abs[abs_col])
        r_eff = df_eff[xcol].corr(df_eff[eff_col])

        n_abs = len(df_abs)
        n_eff = len(df_eff)

        ax.text(
            0.05,
            0.95,
            f"Absolute: $r$ = {r_abs:.2f}, n = {n_abs}\n"
            f"Effective: $r$ = {r_eff:.2f}, n = {n_eff}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        if row == 0:
            ax.set_title(comp["title"])

        if col_idx == 0:
            ax.set_ylabel(f"{dataset['row_label']}\nMIZ width [km]")
        else:
            ax.set_ylabel("")

        if row == 1:
            ax.set_xlabel(comp["xlabel"])
        else:
            ax.set_xlabel("")

        ax.set_ylim(0, 400)
        ax.legend(fontsize=8, loc="upper right")

plt.show()

### Directly comparative scatter plots

In [ ]:
scatter_comparisons = [
    {
        "title": "AltiKa estimate",
        "abs_model_col": inner_col,
        "eff_model_col": inner_col_eff,
        "color": "tab:blue",
    },
    {
        "title": "WW3 estimate",
        "abs_model_col": inner_col_model,
        "eff_model_col": inner_col_model_eff,
        "color": "tab:purple",
    },
    {
        "title": "WW3 estimate ($T_p$)",
        "abs_model_col": "est_miz_width_ww3_tp_km",
        "eff_model_col": "est_miz_width_ww3_tp_eff_km",
        "color": "tab:orange",
    },
    {
        "title": "WW3 estimate ($T_p, \\theta$)",
        "abs_model_col": "est_miz_width_ww3_south_km",
        "eff_model_col": "est_miz_width_ww3_south_eff_km",
        "color": "tab:red",
    },
    {
        "title": "WW3 MIZ width",
        "abs_model_col": "ww3_miz_width_hs05_km_myedge",
        "eff_model_col": "ww3_miz_width_hs05_km_myedge_eff",
        "color": "tab:green",
    },
]

fig, axes = plt.subplots(
    2,
    len(scatter_comparisons),
    figsize=(4 * len(scatter_comparisons), 8),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

df_plot = df_sample

for col_idx, comp in enumerate(scatter_comparisons):
    for row, width_kind in enumerate(["absolute", "effective"]):
        ax = axes[row, col_idx]

        if width_kind == "absolute":
            xcol = obs_col
            ycol = comp["abs_model_col"]
            xlabel = "Observed MIZ width [km]"
            ylabel = "Model MIZ width [km]"
        else:
            xcol = era5_eff_col
            ycol = comp["eff_model_col"]
            xlabel = "Observed effective MIZ width [km]"
            ylabel = "Model effective MIZ width [km]"

        df_valid = (
            df_plot[[xcol, ycol]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        ax.scatter(
            df_valid[xcol],
            df_valid[ycol],
            s=18,
            alpha=0.7,
            color=comp["color"],
            edgecolor="none",
        )

        xy_max = np.nanmax([
            df_valid[xcol].max(),
            df_valid[ycol].max(),
            350,
        ])

        ax.plot(
            [0, xy_max],
            [0, xy_max],
            color="0.4",
            linewidth=1,
            linestyle="--",
        )

        r, rmse, bias, n = relationship_stats(df_valid, xcol, ycol)

        ax.text(
            0.05,
            0.95,
            f"$r$ = {r:.2f}\nRMSE = {rmse:.1f} km\nBias = {bias:.1f} km\nn = {n}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        if row == 0:
            ax.set_title(comp["title"])

        if col_idx == 0:
            ax.set_ylabel(ylabel)
        else:
            ax.set_ylabel("")

        if row == 1:
            ax.set_xlabel(xlabel)
        else:
            ax.set_xlabel("")

        ax.set_xlim(0, 350)
        ax.set_ylim(0, 350)
        ax.set_aspect("equal", adjustable="box")

plt.show()

In [ ]:
df_aligned["ww3_minus_obs"] = (
    df_aligned["ww3_miz_width_hs05_km_myedge"]
    - df_aligned["mizWidthAlongTrackFromMyEdge"]
)

df_aligned["ww3_minus_obs"].describe()

(df_aligned["HS_edge_north_myedge"] < 0.44).mean()

In [ ]:
fig, axes = plt.subplots(
    2,
    len(miz_hs_comparisons),
    figsize=(4 * len(miz_hs_comparisons), 8),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

df_plot = df_sample

for col_idx, comp in enumerate(miz_hs_comparisons):
    xcol = comp["hs_col"]

    for row, width_kind in enumerate(["absolute", "effective"]):
        ax = axes[row, col_idx]

        if width_kind == "absolute":
            ycol = comp["abs_width_col"]
            marker = "o"
            alpha = 0.55
            row_label = "Absolute MIZ width [km]"
            label = comp["abs_label"]
        else:
            ycol = comp["eff_width_col"]
            marker = "^"
            alpha = 0.75
            row_label = "Effective MIZ width [km]"
            label = comp["eff_label"]

        df_valid = (
            df_plot[[xcol, ycol]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        ax.scatter(
            df_valid[xcol],
            df_valid[ycol],
            s=18,
            alpha=alpha,
            color=comp["color"],
            marker=marker,
            edgecolor="none",
            label=label,
        )

        r = df_valid[xcol].corr(df_valid[ycol])
        n = len(df_valid)

        ax.text(
            0.05,
            0.95,
            f"$r$ = {r:.2f}, n = {n}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        if row == 0:
            ax.set_title(comp["title"])

        if col_idx == 0:
            ax.set_ylabel(row_label)
        else:
            ax.set_ylabel("")

        if row == 1:
            ax.set_xlabel(comp["xlabel"])
        else:
            ax.set_xlabel("")

        ax.set_ylim(0, 400)
        ax.legend(fontsize=8, loc="upper right")
ax.set_xlim(0,10)
plt.show()

## Understanding the variance in the observations

Assuming 
$$ H_s^{inner} = H_s^0 e^{-\alpha x} $$
$$ e^{-\alpha x} = \frac{H_s^{inner}}{H_s^0} $$
$$ -\alpha x = \log(H_s^{inner}) - \log(H_s^0) $$
$$ x = \frac{-1}{\alpha}\log(H_s^{inner}) + \frac{1}{\alpha}\log(H_s^0) $$
$$ x = \beta_0 + \beta_1\log(H_s^0) $$

In [ ]:
fig, axes = plt.subplots(
    2,
    len(miz_hs_comparisons),
    figsize=(4 * len(miz_hs_comparisons), 8),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

df_plot = df_sample

for col_idx, comp in enumerate(miz_hs_comparisons):
    xcol = comp["hs_col"]

    for row, width_kind in enumerate(["absolute", "effective"]):
        ax = axes[row, col_idx]

        if width_kind == "absolute":
            ycol = comp["abs_width_col"]
            marker = "o"
            alpha = 0.55
            row_label = "Absolute MIZ width [km]"
            label = comp["abs_label"]
        else:
            ycol = comp["eff_width_col"]
            marker = "^"
            alpha = 0.75
            row_label = "Effective MIZ width [km]"
            label = comp["eff_label"]

        df_valid = (
            df_plot[[xcol, ycol]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        ax.scatter(
            df_valid[xcol],
            df_valid[ycol],
            s=18,
            alpha=alpha,
            color=comp["color"],
            marker=marker,
            edgecolor="none",
            label=label,
        )

        # Need positive Hs and positive MIZ width for fits
        df_fit = df_valid[
            (df_valid[xcol] > 0)
            & (df_valid[ycol] > 0)
        ].copy()
        
        if len(df_fit) >= 3:
            x = df_fit[xcol].to_numpy()
            y = df_fit[ycol].to_numpy()
        
            # Linear fit: y = a + b*x
            slope_lin, intercept_lin = np.polyfit(x, y, 1)
            y_pred_lin = intercept_lin + slope_lin * x
        
            ss_res_lin = np.sum((y - y_pred_lin) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            r2_lin = 1 - ss_res_lin / ss_tot if ss_tot > 0 else np.nan
        
            # Log fit: y = a + b*log(x)
            log_x = np.log(x)
            slope_log, intercept_log = np.polyfit(log_x, y, 1)
            y_pred_log = intercept_log + slope_log * log_x
        
            ss_res_log = np.sum((y - y_pred_log) ** 2)
            r2_log = 1 - ss_res_log / ss_tot if ss_tot > 0 else np.nan
        
            x_fit = np.linspace(x.min(), x.max(), 100)
        
            if r2_log > r2_lin:
                y_fit = intercept_log + slope_log * np.log(x_fit)
                best_label = "best: log($H_s$)"
                best_r2 = r2_log
                best_slope = slope_log
            else:
                y_fit = intercept_lin + slope_lin * x_fit
                best_label = "best: linear"
                best_r2 = r2_lin
                best_slope = slope_lin
        
            ax.plot(
                x_fit,
                y_fit,
                color=comp["color"],
                linewidth=2,
                linestyle="--",
                label=best_label,
            )
        
            fit_text = (
                f"$r$ = {df_valid[xcol].corr(df_valid[ycol]):.2f}, n = {len(df_valid)}\n"
                f"$R^2_{{lin}}$ = {r2_lin:.2f}\n"
                f"$R^2_{{log}}$ = {r2_log:.2f}\n"
                f"{best_label}, $\\beta_1$ = {best_slope:.1f}"
            )
        else:
            fit_text = (
                f"$r$ = {df_valid[xcol].corr(df_valid[ycol]):.2f}, n = {len(df_valid)}\n"
                "fit: insufficient data"
            )

        ax.text(
            0.05,
            0.95,
            fit_text,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        if row == 0:
            ax.set_title(comp["title"])

        if col_idx == 0:
            ax.set_ylabel(row_label)
        else:
            ax.set_ylabel("")

        if row == 1:
            ax.set_xlabel(comp["xlabel"])
        else:
            ax.set_xlabel("")

        ax.set_ylim(0, 400)
        ax.set_xlim(0, 10)
        ax.legend(fontsize=8, loc="upper right")

plt.show()

In [ ]:
def log_hs_fit_sums(df, xcol, ycol):
    df_fit = (
        df[[xcol, ycol]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    df_fit = df_fit[
        (df_fit[xcol] > 0)
        & (df_fit[ycol] > 0)
    ].copy()

    if len(df_fit) < 3:
        return {
            "n": len(df_fit),
            "intercept": np.nan,
            "slope": np.nan,
            "ss_res": np.nan,
            "ss_tot": np.nan,
            "sample_var": np.nan,
            "resid_var": np.nan,
            "s2": np.nan,
            "r2": np.nan,
        }

    x = df_fit[xcol].to_numpy()
    y = df_fit[ycol].to_numpy()
    log_x = np.log(x)

    slope, intercept = np.polyfit(log_x, y, 1)

    y_pred = intercept + slope * log_x
    resid = y - y_pred

    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)

    sample_var = ss_tot / (len(y) - 1)
    resid_var = np.var(resid, ddof=1)
    s2 = ss_res / (len(y) - 2)

    r2 = 1 - ss_res / ss_tot

    return {
        "n": len(df_fit),
        "intercept": intercept,
        "slope": slope,
        "ss_res": ss_res,
        "ss_tot": ss_tot,
        "sample_var": sample_var,
        "resid_var": resid_var,
        "s2": s2,
        "r2": r2,
    }

In [ ]:
absolute_fit_summary = pd.DataFrame([
    {
        "case": comp["title"],
        "color": comp["color"],
        "xcol": comp["hs_col"],
        "ycol": comp["abs_width_col"],
        **log_hs_fit_sums(
            df_sample,
            xcol=comp["hs_col"],
            ycol=comp["abs_width_col"],
        ),
    }
    for comp in miz_hs_comparisons
])

In [ ]:
bar_metrics = [
    "intercept",
    "slope",
    "ss_res",
    "ss_tot",
    "sample_var",
    "resid_var",
    "s2",
    "r2",
]

fig, axes = plt.subplots(
    2,
    4,
    figsize=(18, 8),
    constrained_layout=True,
)

axes = axes.ravel()

for ax, metric in zip(axes, bar_metrics):
    ax.bar(
        absolute_fit_summary["case"],
        absolute_fit_summary[metric],
        color=absolute_fit_summary["color"],
    )

    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=45)

plt.show()

In [ ]:
def conditional_bin_stats(df, xcol, ycol, bins, interval="iqr"):
    df_valid = (
        df[[xcol, ycol]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )

    # df_valid = df_valid[df_valid[xcol] > 0].copy()
    df_valid = df_valid[
        (df_valid[xcol] > 0)
        & (df_valid[ycol] > 0)
    ].copy()
    df_valid["x_bin"] = pd.cut(df_valid[xcol], bins=bins)

    out = (
        df_valid
        .groupby("x_bin", observed=True)[ycol]
        .agg(
            n="count",
            mean="mean",
            median="median",
            std="std",
            var="var",
            q25=lambda x: x.quantile(0.25),
            q75=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )

    out["x_mid"] = out["x_bin"].apply(lambda b: b.mid).astype(float)

    if interval == "iqr":
        out["low"] = out["q25"]
        out["high"] = out["q75"]
        out["interval_label"] = "IQR"

    elif interval == "ci":
        z = 1.96
        out["sem"] = out["std"] / np.sqrt(out["n"])
        out["low"] = out["mean"] - z * out["sem"]
        out["high"] = out["mean"] + z * out["sem"]
        out["interval_label"] = "95% CI"

    else:
        raise ValueError("interval must be 'iqr' or 'ci'")

    return out

In [ ]:
def compare_conditional_to_observed(
    df,
    comparisons,
    width_kind="absolute",
    bins=None,
):
    if bins is None:
        bins = np.arange(0, 10.5, 0.5)  # change if you want different Hs bins
        
    if width_kind == "absolute":
        obs_ycol = obs_col
        model_ykey = "abs_width_col"
    elif width_kind == "effective":
        obs_ycol = era5_eff_col
        model_ykey = "eff_width_col"
    else:
        raise ValueError("width_kind must be 'absolute' or 'effective'")

    obs_cond = conditional_bin_stats(
        df,
        xcol="hs_my_edge",
        ycol=obs_ycol,
        bins=bins,
    )

    rows = []

    for comp in comparisons:
        model_cond = conditional_bin_stats(
            df,
            xcol=comp["hs_col"],
            ycol=comp[model_ykey],
            bins=bins,
        )

        merged = obs_cond.merge(
            model_cond,
            on="x_mid",
            suffixes=("_obs", "_model"),
        ).dropna()

        rows.append({
            "case": comp["title"],
            "width_kind": width_kind,
            "n_bins": len(merged),
            "mean_abs_mean_diff": np.mean(np.abs(merged["mean_model"] - merged["mean_obs"])),
            "rmse_mean_diff": np.sqrt(np.mean((merged["mean_model"] - merged["mean_obs"]) ** 2)),
            "mean_bias": np.mean(merged["mean_model"] - merged["mean_obs"]),
            "mean_std_ratio": np.mean(merged["std_model"] / merged["std_obs"]),
            "mean_var_ratio": np.mean(merged["var_model"] / merged["var_obs"]),
        })

    return pd.DataFrame(rows)

In [ ]:
conditional_abs_summary = compare_conditional_to_observed(
    df_sample,
    miz_hs_comparisons,
    width_kind="absolute",
)

conditional_eff_summary = compare_conditional_to_observed(
    df_sample,
    miz_hs_comparisons,
    width_kind="effective",
)

conditional_summary = pd.concat(
    [conditional_abs_summary, conditional_eff_summary],
    ignore_index=True,
)

conditional_summary

In [ ]:
estimate_comparisons = [
    comp for comp in miz_hs_comparisons
    if comp["title"] != "AltiKa observed"
]

In [ ]:
conditional_abs_summary = compare_conditional_to_observed(
    df_sample,
    estimate_comparisons,
    width_kind="absolute",
)

conditional_eff_summary = compare_conditional_to_observed(
    df_sample,
    estimate_comparisons,
    width_kind="effective",
)

In [ ]:
def plot_conditional_relationships(
    df,
    comparisons,
    width_kind="absolute",
    interval="iqr",
    bins=None,
):
    if bins is None:
        bins = np.arange(0, 10.5, 0.5)  # change if you want different Hs bins

    if width_kind == "absolute":
        obs_ycol = obs_col
        model_ykey = "abs_width_col"
        ylabel = "Absolute MIZ width [km]"
        title = "Conditional absolute MIZ width by $H_s$"
    elif width_kind == "effective":
        obs_ycol = era5_eff_col
        model_ykey = "eff_width_col"
        ylabel = "Effective MIZ width [km]"
        title = "Conditional effective MIZ width by $H_s$"
    else:
        raise ValueError("width_kind must be 'absolute' or 'effective'")

    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

    obs_cond = conditional_bin_stats(
        df,
        xcol="hs_my_edge",
        ycol=obs_ycol,
        bins=bins,
        interval=interval,
    )

    interval_label = obs_cond["interval_label"].iloc[0]

    ax.plot(
        obs_cond["x_mid"],
        obs_cond["mean"],
        color="black",
        linewidth=2.5,
        label="Observed mean",
    )

    ax.fill_between(
        obs_cond["x_mid"],
        obs_cond["low"],
        obs_cond["high"],
        color="black",
        alpha=0.15,
        label=f"Observed {interval_label}",
    )

    for comp in comparisons:
        if comp["title"] == "AltiKa observed":
            continue

        model_cond = conditional_bin_stats(
            df,
            xcol=comp["hs_col"],
            ycol=comp[model_ykey],
            bins=bins,
            interval=interval,
        )

        ax.plot(
            model_cond["x_mid"],
            model_cond["mean"],
            color=comp["color"],
            linewidth=2,
            label=f"{comp['title']} mean",
        )

        ax.fill_between(
            model_cond["x_mid"],
            model_cond["low"],
            model_cond["high"],
            color=comp["color"],
            alpha=0.12,
        )

    ax.set_xlabel("$H_s$ [m]")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title} ({interval_label})")
    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, 600)
    ax.legend(fontsize=8)

    return fig, ax

In [ ]:
fig_abs, ax_abs = plot_conditional_relationships(
    df_sample,
    miz_hs_comparisons,
    width_kind="absolute",
)
plt.show()

In [ ]:
fig_eff, ax_eff = plot_conditional_relationships(
    df_sample,
    miz_hs_comparisons,
    width_kind="effective",
)
plt.show()

### Statistically analysis the variance from $T_p$

In [ ]:
def residual_spread_stats(df, xcol, ycol, degree=1):
    df_valid = (
        df[[xcol, ycol]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(df_valid) < degree + 2:
        return {
            "n": len(df_valid),
            "r": np.nan,
            "resid_std": np.nan,
            "resid_var": np.nan,
            "resid_iqr": np.nan,
        }

    x = df_valid[xcol].to_numpy()
    y = df_valid[ycol].to_numpy()

    coef = np.polyfit(x, y, degree)
    y_fit = np.polyval(coef, x)
    resid = y - y_fit

    return {
        "n": len(df_valid),
        "r": df_valid[xcol].corr(df_valid[ycol]),
        "resid_std": np.std(resid, ddof=1),
        "resid_var": np.var(resid, ddof=1),
        "resid_iqr": np.percentile(resid, 75) - np.percentile(resid, 25),
    }

In [ ]:
def compare_tp_spread(df, width_type="effective", degree=1):
    if width_type == "absolute":
        obs_width = obs_col
        no_tp_width = inner_col_model
        tp_width = "est_miz_width_ww3_tp_km"
    elif width_type == "effective":
        obs_width = era5_eff_col
        no_tp_width = inner_col_model_eff
        tp_width = "est_miz_width_ww3_tp_eff_km"
    else:
        raise ValueError("width_type must be 'absolute' or 'effective'")

    rows = []

    comparisons = {
        "Observed": {
            "xcol": "hs_my_edge",
            "ycol": obs_width,
        },
        "WW3 estimate, no Tp": {
            "xcol": "hs",
            "ycol": no_tp_width,
        },
        "WW3 estimate, with Tp": {
            "xcol": "hs",
            "ycol": tp_width,
        },
    }

    for name, cols in comparisons.items():
        stats = residual_spread_stats(
            df,
            cols["xcol"],
            cols["ycol"],
            degree=degree,
        )

        rows.append({
            "case": name,
            "width_type": width_type,
            **stats,
        })

    out = pd.DataFrame(rows)

    no_tp_var = out.loc[out["case"] == "WW3 estimate, no Tp", "resid_var"].iloc[0]
    tp_var = out.loc[out["case"] == "WW3 estimate, with Tp", "resid_var"].iloc[0]
    obs_var = out.loc[out["case"] == "Observed", "resid_var"].iloc[0]

    tp_added_var = tp_var - no_tp_var

    out["tp_added_var"] = np.nan
    out["tp_added_std_equiv"] = np.nan
    out["fraction_of_observed_spread"] = np.nan

    out.loc[out["case"] == "WW3 estimate, with Tp", "tp_added_var"] = tp_added_var
    out.loc[out["case"] == "WW3 estimate, with Tp", "tp_added_std_equiv"] = np.sqrt(max(tp_added_var, 0))
    out.loc[out["case"] == "WW3 estimate, with Tp", "fraction_of_observed_spread"] = tp_added_var / obs_var

    return out

In [ ]:
spread_date_eff = compare_tp_spread(df_by_date, width_type="effective")
spread_lon_eff = compare_tp_spread(df_by_lon, width_type="effective")

spread_date_abs = compare_tp_spread(df_by_date, width_type="absolute")
spread_lon_abs = compare_tp_spread(df_by_lon, width_type="absolute")

spread_date_eff

In [ ]:
spread_summary = pd.concat(
    {
        "date_absolute": spread_date_abs,
        "date_effective": spread_date_eff,
        "lon_absolute": spread_lon_abs,
        "lon_effective": spread_lon_eff,
    },
    names=["dataset"],
).reset_index(level=0)

spread_summary

In [ ]:
spread_captured_no_tp = 138.646038 / 517.463053
spread_captured_with_tp = 301.051914 / 517.463053

print(spread_captured_no_tp, spread_captured_with_tp)

In [ ]:
import statsmodels.api as sm

def fit_hs_tp_model(df, ycol, hs_col="hs", tp_col="tp"):
    df_valid = (
        df[[ycol, hs_col, tp_col]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    X = df_valid[[hs_col, tp_col]]
    X = sm.add_constant(X)

    y = df_valid[ycol]

    model = sm.OLS(y, X).fit()

    return model

In [ ]:
model_obs_eff = fit_hs_tp_model(
    df_by_date,
    ycol="mizWidthAlongTrackFromMyEdge_eff_era5",
    hs_col="hs_my_edge",  # or "hs", depending what you want as predictor
    tp_col="tp",
)

print(model_obs_eff.summary())

In [ ]:
def fit_linear_model(df, ycol, xcols):
    df_valid = (
        df[[ycol] + xcols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    X = sm.add_constant(df_valid[xcols])
    y = df_valid[ycol]

    return sm.OLS(y, X).fit()
df_sample.columns

$$ \hat{MIZ} \sim \beta_0 + \beta_1 H_s + \beta_2 T_p $$

In [ ]:
ycol = "est_miz_width_ww3_tp_km"
hs_col = "hs"
tp_col = "tp"
# ycol="mizWidthAlongTrackFromMyEdge_eff_era5"
# hs_col="hs_my_edge"

model_hs_only = fit_linear_model(
    df_sample,
    ycol=ycol,
    xcols=[hs_col],
)

model_hs_tp = fit_linear_model(
    df_sample,
    ycol=ycol,
    xcols=[hs_col, tp_col],
)

print("Hs only R²:", model_hs_only.rsquared)
print("Hs + Tp R²:", model_hs_tp.rsquared)

In [ ]:
df_valid = (
    df_by_date[[ycol, hs_col, tp_col]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

df_valid["pred_hs"] = model_hs_only.predict(
    sm.add_constant(df_valid[[hs_col]])
)

df_valid["pred_hs_tp"] = model_hs_tp.predict(
    sm.add_constant(df_valid[[hs_col, tp_col]])
)

plt.figure(figsize=(5, 5))

plt.scatter(
    df_valid[ycol],
    df_valid["pred_hs"],
    alpha=0.5,
    label="$H_s$ only",
)

plt.scatter(
    df_valid[ycol],
    df_valid["pred_hs_tp"],
    alpha=0.5,
    label="$H_s + T_p$",
)

lims = [
    0,
    max(
        df_valid[ycol].max(),
        df_valid["pred_hs"].max(),
        df_valid["pred_hs_tp"].max(),
    ),
]

plt.plot(lims, lims, "k--", linewidth=1)

plt.text(
    0.05,
    0.95,
    f"$H_s$ only: $R^2$ = {model_hs_only.rsquared:.2f}\n"
    f"$H_s + T_p$: $R^2$ = {model_hs_tp.rsquared:.2f}",
    transform=plt.gca().transAxes,
    ha="left",
    va="top",
    bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
)

plt.xlabel("Target WW3 $T_p$ MIZ width [km]")
plt.ylabel("Predicted WW3 $T_p$ MIZ width [km]")
plt.legend()
plt.show()

In [ ]:
# era5_eff_col

In [ ]:
# tp_model = fit_linear_model(
#     df_sample,
#     ycol="tp",
#     xcols=["hs_my_edge", "mizWidthAlongTrackFromMyEdge"],
# )

# print(tp_model.summary())

In [ ]:
# # This is the model you already fit:
# # y = est_miz_width_ww3_tp_km
# # x = hs, tp

params = model_hs_tp.params

b0 = params["const"]
b_hs = params["hs"]
b_tp = params["tp"]

b0, b_hs, b_tp

In [ ]:
df_sample.columns

$$ \hat{MIZ} \sim \beta_0 + \beta_1 H_s + \beta_2 T_p $$
$$ T_p = \frac{-\beta_0 - \beta_1 H_s + \hat{MIZ}}{\beta_2 }  $$

In [ ]:
df_sample["tp_pred_from_altika"] = np.nan

mask = (
    df_sample[["mizWidthAlongTrackFromMyEdge", "hs_my_edge"]]
    .replace([np.inf, -np.inf], np.nan)
    .notna()
    .all(axis=1)
)

df_sample.loc[mask, "tp_pred_from_altika"] = (
    df_sample.loc[mask, "mizWidthAlongTrackFromMyEdge"]
    - b0
    - b_hs * df_sample.loc[mask, "hs_my_edge"]
) / b_tp

In [ ]:
from scipy.optimize import brentq

def estimate_tp_from_hs_width_target(
    hs0,
    mizwidth_km,
    target_hs,
    tp_min=4,
    tp_max=25,
    aice=1.0,
    thickness=0.5,
    floeDiameter=1000,
):
    if pd.isna(hs0) or pd.isna(mizwidth_km) or pd.isna(target_hs):
        return np.nan

    if hs0 <= 0 or mizwidth_km < 0 or target_hs <= 0:
        return np.nan

    def objective(tp):
        return hs_after_distance(
            hs0=hs0,
            distance_km=mizwidth_km,
            tp=tp,
            aice=aice,
            thickness=thickness,
            floeDiameter=floeDiameter,
        ) - target_hs

    f_min = objective(tp_min)
    f_max = objective(tp_max)

    if pd.isna(f_min) or pd.isna(f_max):
        return np.nan

    # No crossing in the allowed Tp range
    if f_min * f_max > 0:
        return np.nan

    return brentq(objective, tp_min, tp_max)

In [ ]:
df_sample["tp_inferred_propagation"] = df_sample.apply(
    lambda row: estimate_tp_from_hs_width_target(
        hs0=row["hs_my_edge"],
        mizwidth_km=row["mizWidthAlongTrackFromMyEdge"],
        target_hs=target_hs,
        tp_min=4,
        tp_max=25,
    ),
    axis=1,
)

In [ ]:
# obs_col

In [ ]:
tp_plot = (
    df_sample[[
        "tp",
        "tp_pred_from_altika",
        "tp_inferred_propagation",
        # "tp_equiv_altika_from_mizwidth_eff",
        # "tp_equiv_altika_from_log_mizwidth",
    ]]
    .replace([np.inf, -np.inf], np.nan)
)

plt.figure(figsize=(6, 4))

sns.kdeplot(
    tp_plot["tp"].dropna(),
    linewidth=2,
    label="WW3 $T_p$",
)

sns.kdeplot(
    tp_plot["tp_pred_from_altika"].dropna(),
    linewidth=2,
    label="Linear model",
)

sns.kdeplot(
    tp_plot["tp_inferred_propagation"].dropna(),
    linewidth=2,
    label="Inferred",
)

# sns.kdeplot(
#     tp_plot["tp_equiv_altika_from_log_mizwidth"].dropna(),
#     linewidth=2,
#     label="Log-width model",
# )
plt.xlim(2.5,20)
plt.xlabel("$T_p$ [s]")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
obs_col = "mizWidthAlongTrackFromMyEdge"
inner_col = "est_miz_width_altika_km"
inner_model_col = "est_miz_width_ww3_km"
era5_eff_col = "mizWidthAlongTrackFromMyEdge_eff_era5"
inner_col_eff = "est_miz_width_altika_eff_km"
inner_col_model = "est_miz_width_ww3_km"
inner_col_model_eff = "est_miz_width_ww3_eff_km"

value_cols = [
    obs_col,
    inner_col,
    inner_col_eff,
    inner_model_col,
    inner_col_model_eff,
    era5_eff_col,
    "est_miz_width_ww3_tp_km",
    "est_miz_width_ww3_tp_eff_km",
    "est_miz_width_ww3_south_km",
    "est_miz_width_ww3_south_eff_km",
    "mean_siconc_myedge_track",
    "tp",
    "tp_pred_from_altika",
    "tp_inferred_propagation",
    "hs",
    "hs_my_edge",
    "mizwidth",
    "mizwidth_eff"
]



df_line = (
    df_sample[["time", "lon"] + value_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

df_line["time"] = pd.to_datetime(df_line["time"])
df_line["date"] = df_line["time"].dt.floor("D")
df_line["lon_bin"] = np.floor(df_line["lon"] % 360).astype(int)

df_by_date = (
    df_line
    .groupby("date", as_index=False)[value_cols]
    .mean()
)

df_by_lon = (
    df_line
    .groupby("lon_bin", as_index=False)[value_cols]
    .mean()
)

In [ ]:
def corr_text(df, xcol, ycol):
    df_valid = df[[xcol, ycol]].replace([np.inf, -np.inf], np.nan).dropna()
    r = df_valid[xcol].corr(df_valid[ycol])
    return f"r = {r:.2f}\nn = {len(df_valid)}"

In [ ]:
plot_cols = [
    "mean_siconc_myedge_track",
    "tp",
    "hs",
]

labels = {
    "mean_siconc_myedge_track": "Mean sea-ice concentration",
    "tp": "$T_p$ [s]",
    "hs": "$H_s$ [m]",
    "hs_my_edge": "Altimeter $H_s$ at outer edge [m]",
    "tp_inferred_propagation": "Predicted $T_p$ [s]",
}

fig, axes = plt.subplots(
    len(plot_cols),
    2,
    figsize=(12, 3 * len(plot_cols)),
    constrained_layout=True
)

for i, col in enumerate(plot_cols):
    # Date panel
    ax = axes[i, 0]

    ax.plot(
        df_by_date["date"],
        df_by_date[col],
        linewidth=1.2,
        label=labels[col],
    )

    if col == "hs":
        ax.plot(
            df_by_date["date"],
            df_by_date["hs_my_edge"],
            linewidth=1.2,
            label=labels["hs_my_edge"],
        )

        ax.text(
            0.02,
            0.95,
            corr_text(df_by_date, "hs", "hs_my_edge"),
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )
        
    if col == "tp":
        ax.plot(
            df_by_date["date"],
            df_by_date["tp_inferred_propagation"],
            linewidth=1.2,
            label=labels["tp_inferred_propagation"],
        )

        ax.text(
            0.02,
            0.95,
            corr_text(df_by_date, "tp", "tp_inferred_propagation"),
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        ax.legend(loc="upper right")

    ax.set_xlabel("Date")
    ax.set_ylabel(labels[col])
    ax.set_title(f"{labels[col]} by date")

    # Longitude panel
    ax = axes[i, 1]

    ax.plot(
        df_by_lon["lon_bin"],
        df_by_lon[col],
        linewidth=1.2,
        label=labels[col],
    )

    if col == "hs":
        ax.plot(
            df_by_lon["lon_bin"],
            df_by_lon["hs_my_edge"],
            linewidth=1.2,
            label=labels["hs_my_edge"],
        )

        ax.text(
            0.02,
            0.95,
            corr_text(df_by_lon, "hs", "hs_my_edge"),
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        ax.legend(loc="upper right")

    if col == "tp":
        ax.plot(
            df_by_lon["lon_bin"],
            df_by_lon["tp_inferred_propagation"],
            linewidth=1.2,
            label=labels["tp_inferred_propagation"],
        )

        ax.text(
            0.02,
            0.95,
            corr_text(df_by_lon, "tp", "tp_inferred_propagation"),
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"),
        )

        ax.legend(loc="upper right")

    ax.set_xlabel("Longitude [$^\circ$E]")
    ax.set_ylabel(labels[col])
    ax.set_title(f"{labels[col]} by longitude")
    ax.set_xlim(0, 359)

plt.show()

In [ ]:
tp_date = df_by_date[["date", "tp", "tp_inferred_propagation"]].dropna().copy()
tp_lon = df_by_lon[["lon_bin", "tp", "tp_inferred_propagation"]].dropna().copy()

tp_date = tp_date.sort_values("date")
tp_lon = tp_lon.sort_values("lon_bin")

date_window = 7      # days / date bins
lon_window = 10      # longitude bins

tp_date["tp_rolling"] = (
    tp_date["tp"]
    .rolling(window=date_window, center=True, min_periods=1)
    .mean()
)

tp_lon["tp_rolling"] = (
    tp_lon["tp"]
    .rolling(window=lon_window, center=True, min_periods=1)
    .mean()
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4),
    constrained_layout=True,
)

# Date panel
ax = axes[0]
ax.plot(
    tp_date["date"],
    tp_date["tp"],
    color="0.75",
    linewidth=1,
    label="$T_p$",
)

ax.plot(
    tp_date["date"],
    tp_date["tp_rolling"],
    color="tab:blue",
    linewidth=2,
    label=f"{date_window}-point rolling mean",
)

ax.plot(
    tp_date["date"],
    tp_date["tp_inferred_propagation"],
    color="tab:red",
    linewidth=1,
    label="$T_p$",
)

ax.set_xlabel("Date")
ax.set_ylabel("$T_p$ [s]")
ax.set_title("$T_p$ by date")
ax.legend()

# Longitude panel
ax = axes[1]
ax.plot(
    tp_lon["lon_bin"],
    tp_lon["tp"],
    color="0.75",
    linewidth=1,
    label="$T_p$",
)

ax.plot(
    tp_lon["lon_bin"],
    tp_lon["tp_rolling"],
    color="tab:blue",
    linewidth=2,
    label=f"{lon_window}-point rolling mean",
)

ax.plot(
    tp_lon["lon_bin"],
    tp_lon["tp_inferred_propagation"],
    color="tab:red",
    linewidth=1,
    label="$T_p$",
)

ax.set_xlabel("Longitude [$^\circ$E]")
ax.set_ylabel("$T_p$ [s]")
ax.set_title("$T_p$ by longitude")
ax.set_xlim(0, 359)
ax.legend()

plt.show()

### Read in aligned dataset

In [ ]:
df_aligned = pd.read_csv("/g/data/ps29/nd0349/Fraser-2024/data/aligned/df_aligned_20130314_to_20150401.csv")
df_aligned

In [ ]:
x = df_aligned["tp"].dropna()

# Histogram overlap coefficient
bins = np.linspace(
    x.min(),
    25
)

px, _ = np.histogram(x, bins=bins, density=True)
bin_width = np.diff(bins)

plt.figure(figsize=(8, 5))

sns.histplot(
    x,
    bins=bins,
    stat="density",
    color="tab:blue",
    edgecolor=None,
    linewidth=0,
    alpha=0.25
)
sns.kdeplot(x, color="tab:blue", label="WW3 $H_s$")


In [ ]:
plt.scatter(df_aligned["tp"], df_aligned["hs"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

hs_col = "hs"
tp_col = "tp"

df_fit = df_aligned[[hs_col, tp_col]].replace([np.inf, -np.inf], np.nan).dropna()
df_fit = df_fit[(df_fit[hs_col] > 0) & (df_fit[tp_col] > 0)]

x = df_fit[hs_col].values
y = df_fit[tp_col].values

In [ ]:
def linear_model(hs, a, b):
    return a + b * hs

def power_law(hs, a, b):
    return a * hs**b

# Fit models
popt_linear, _ = curve_fit(linear_model, x, y)
popt_power, _ = curve_fit(power_law, x, y, p0=[10, 0.3], maxfev=10000)

# Predictions
y_pred_linear = linear_model(x, *popt_linear)
y_pred_power = power_law(x, *popt_power)

In [ ]:
def fit_stats(y_true, y_pred, n_params):
    n = len(y_true)
    residuals = y_true - y_pred
    rss = np.sum(residuals**2)

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    # AIC/BIC assuming Gaussian residuals
    aic = n * np.log(rss / n) + 2 * n_params
    bic = n * np.log(rss / n) + n_params * np.log(n)

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "AIC": aic,
        "BIC": bic,
    }

stats_linear = fit_stats(y, y_pred_linear, n_params=2)
stats_power = fit_stats(y, y_pred_power, n_params=2)

print("Linear model")
print(f"Tp = {popt_linear[0]:.3f} + {popt_linear[1]:.3f} Hs")
print(stats_linear)

print("\nPower-law model")
print(f"Tp = {popt_power[0]:.3f} Hs^{popt_power[1]:.3f}")
print(stats_power)

In [ ]:
hs_grid = np.linspace(x.min(), x.max(), 300)

tp_linear = linear_model(hs_grid, *popt_linear)
tp_power = power_law(hs_grid, *popt_power)

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(x, y, s=10, alpha=0.25, label="Data")

ax.plot(
    hs_grid,
    tp_linear,
    color="tab:red",
    linewidth=2,
    label=f"Linear: $T_p={popt_linear[0]:.2f}+{popt_linear[1]:.2f}H_s$"
)

ax.plot(
    hs_grid,
    tp_power,
    color="tab:blue",
    linewidth=2,
    label=f"Power: $T_p={popt_power[0]:.2f}H_s^{{{popt_power[1]:.2f}}}$"
)

stats_text = (
    "Linear\n"
    f"$R^2$ = {stats_linear['R2']:.3f}\n"
    f"RMSE = {stats_linear['RMSE']:.2f} s\n"
    f"MAE = {stats_linear['MAE']:.2f} s\n"
    f"AIC = {stats_linear['AIC']:.1f}\n\n"
    "Power law\n"
    f"$R^2$ = {stats_power['R2']:.3f}\n"
    f"RMSE = {stats_power['RMSE']:.2f} s\n"
    f"MAE = {stats_power['MAE']:.2f} s\n"
    f"AIC = {stats_power['AIC']:.1f}"
)

ax.text(
    0.97,
    0.03,
    stats_text,
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    bbox=dict(facecolor="white", alpha=0.85, edgecolor="none")
)

ax.set_xlabel("$H_s$ [m]")
ax.set_ylabel("$T_p$ [s]")
ax.legend()
ax.grid(True)

plt.show()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score

In [ ]:
df_ml = df_aligned[
    ["time", "aice", "hs", "mwd", "lat", "lon", "mizwidth", "tp"]
].replace([np.inf, -np.inf], np.nan).dropna().copy()

df_ml["time"] = pd.to_datetime(df_ml["time"])

df_ml["dayofyear"] = df_ml["time"].dt.dayofyear
df_ml["doy_sin"] = np.sin(2 * np.pi * df_ml["dayofyear"] / 366)
df_ml["doy_cos"] = np.cos(2 * np.pi * df_ml["dayofyear"] / 366)

df_ml["mwd_sin"] = np.sin(df_ml["mwd"])
df_ml["mwd_cos"] = np.cos(df_ml["mwd"])

df_ml["lon_sin"] = np.sin(np.deg2rad(df_ml["lon"]))
df_ml["lon_cos"] = np.cos(np.deg2rad(df_ml["lon"]))

In [ ]:
candidate_features = [
    "aice",
    "hs",
    "lat",
    "mizwidth",
    "doy_sin",
    "doy_cos",
    "lon_sin",
    "lon_cos",
]

X_all = df_ml[candidate_features]
y = df_ml["tp"]

In [ ]:
def forward_stepwise_selection(X, y, cv=5):
    remaining = list(X.columns)
    selected = []
    history = []

    kfold = KFold(n_splits=cv, shuffle=True, random_state=42)

    best_rmse = np.inf

    while remaining:
        scores = []

        for feature in remaining:
            trial_features = selected + [feature]

            model = make_pipeline(
                StandardScaler(),
                LinearRegression()
            )

            neg_mse_scores = cross_val_score(
                model,
                X[trial_features],
                y,
                cv=kfold,
                scoring="neg_mean_squared_error"
            )

            rmse = np.sqrt(-neg_mse_scores.mean())

            scores.append({
                "feature_added": feature,
                "features": trial_features,
                "cv_rmse": rmse,
            })

        scores_df = pd.DataFrame(scores).sort_values("cv_rmse")
        best_candidate = scores_df.iloc[0]

        if best_candidate["cv_rmse"] < best_rmse:
            selected.append(best_candidate["feature_added"])
            remaining.remove(best_candidate["feature_added"])
            best_rmse = best_candidate["cv_rmse"]

            history.append({
                "step": len(selected),
                "feature_added": best_candidate["feature_added"],
                "selected_features": selected.copy(),
                "cv_rmse": best_rmse,
            })
        else:
            break

    return selected, pd.DataFrame(history)

In [ ]:
selected_features, stepwise_history = forward_stepwise_selection(X_all, y, cv=5)

print("Selected features:")
print(selected_features)

print("\nStepwise history:")
print(stepwise_history)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

X = df_ml[selected_features]
y = df_ml["tp"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

final_model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"Final selected model")
print(f"Features: {selected_features}")
print(f"R2 = {r2:.3f}")
print(f"RMSE = {rmse:.2f} s")
print(f"MAE = {mae:.2f} s")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(
    stepwise_history["step"],
    stepwise_history["cv_rmse"],
    marker="o"
)

for _, row in stepwise_history.iterrows():
    ax.text(
        row["step"],
        row["cv_rmse"],
        row["feature_added"],
        ha="left",
        va="bottom",
        fontsize=9
    )

ax.set_xlabel("Step")
ax.set_ylabel("Cross-validated RMSE [s]")
ax.grid(True)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(y_test, y_pred, s=12, alpha=0.3)

lims = [
    min(y_test.min(), y_pred.min()),
    max(y_test.max(), y_pred.max()),
]

ax.plot(lims, lims, "k--", linewidth=1)

ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Observed $T_p$ [s]")
ax.set_ylabel("Fitted $T_p$ [s]")

ax.text(
    0.04,
    0.96,
    f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.2f} s\nMAE = {mae:.2f} s\nn = {len(y_test)}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    bbox=dict(facecolor="white", alpha=0.85, edgecolor="none")
)

ax.grid(True)

plt.show()

In [ ]:
# df_aligned["Tp_swell"] = estimate_tp_swell_from_hs(df_aligned["hs_my_edge"])
# df_aligned["hs_inner_edge"] = df_sample.apply(
#     lambda row: hs_after_distance(
#         hs0=row["hs_my_edge"],
#         distance_km=row["mizWidthAlongTrackFromMyEdge"],
#         tp=row["Tp_swell"],
#     ),
#     axis=1
# )

### 

### 

In [ ]:
client.close()